In [ ]:
import os
import sys
import warnings
import time
import json
from natsort import natsorted
from pathlib import PureWindowsPath, PurePosixPath
import pickle
import numpy as np
import xarray as xr
import pandas as pd
import math 

import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.pyplot import figure
from matplotlib.patches import Patch, Rectangle
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from matplotlib.legend_handler import HandlerTuple
import seaborn as sns
import cmasher as cmr

from scipy.signal import find_peaks, peak_widths
import scipy.stats as stats
from scipy.stats import skew, median_abs_deviation
from scipy.spatial.distance import cosine, euclidean
import scikit_posthocs as sp
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.decomposition import PCA
from scipy.ndimage import gaussian_filter1d, median_filter
from fastdtw import fastdtw

sys.path.append('../utils') 
from utils_tfc import TFC_proto
from utils_tfc import open_minian, xrconcat_recursive, map_ts
from utils_plot import (
    set_pub_style, get_asterisks, save_metadata_json, 
    lighten_color, add_stat_annotation_two_sided)

#General parameters
dpath_cal_all = r'../../data/11.Post_TFC-20' # The directory for TFC-20
bin_width = 200  # ms
fs = int(1000/bin_width)

colors_anatomy = ['#A6761D', '#845ec2', '#97cebf'] 
colors_beh_i = ['#4091cf', '#e1703c'] # Blue, red
colors_beh_e = ['#4091cf', '#8cba54'] # Blue, green
#plot
dir_output = r'../output_figures'
os.makedirs(dir_output, exist_ok=True)
dir_fig = 'Fig6'
dpath_plot = os.path.join(dir_output, dir_fig)
if not os.path.exists(dpath_plot):
    os.makedirs(dpath_plot)   

## 1.0 Speed plotting of conditioning session

In [ ]:
# Speed of each trial comparision
group_name = ['02.CA1-C', '03.CA1-I']
group_keys = ['CA1-C', 'CA1-I'] 
group_size = len(group_name)

test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

width_mm = 50  # 
height_mm = 30 #  
epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'P_US1': 3, 'P_US2': 6}

base_du = 3 
post_du = 40 
bins_trial_start = int((20-base_du)*fs) 
bins_trial_end = int((20+20+20+3+ post_du)*fs)
bins_speed_trial = int((20+160)*fs) # 20s base + 160 trail

for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_test) 
     # Read in speed info
    with open(os.path.join(dpath_test, 'Speed_bin_trials.pkl'), 'rb') as f:
        speed_bin_trial = pickle.load(f)   
    speed_mean_anmials = []
    for key in speed_bin_trial.keys():
        speed_tmp = speed_bin_trial[key] # Average all trials
        tmp = np.full((6, bins_speed_trial), np.nan) # TO take care of ITI length
        tmp[:, :speed_tmp.shape[-1]] = speed_tmp# Average from 6 trials
        speed_mean_anmials.append(tmp[:, bins_trial_start:bins_trial_end])
    speed_trial_animal = np.stack(speed_mean_anmials)
    plot_speed_trial_to_trial(speed_trial_animal, 6, 'Speed (cm/s)', width_mm, height_mm, epochs, colors_beh_i[i],
                      fs, dpath_plot, f'sup_01_0_Speed-animal-wise-all_trials_{group_keys[i]}')
    
print('All finished************')     

In [ ]:
def plot_speed_trial_to_trial(speed_trial, n_trials, y_label, width_mm, height_mm, epochs, colors, fs, output_path, title):
    """
    Plots the average speed from Trial 1 to Trial 6 using a broken Y-axis 
    to accommodate the massive US escape response and the subtle CS/Trace freezing dynamics.
    
    speed_trial: 3D np.array of shape (animal_n, trial_n, bins_n)
    """
    set_pub_style()    
    # 1. SETUP BROKEN AXES (Top for US burst, Bottom for CS/Trace dynamics)
    fig, (ax_top, ax_bottom) = plt.subplots(
        2, 1, 
        figsize=(width_mm / 25.4, height_mm / 25.4), 
        layout='constrained', # Ensures labels don't get cut off
        sharex=True,
        # Decrease 'hspace' to shrink the blank space between the top and bottom panels. 
        # 0.05 is very narrow; 0.0 makes them physically touch.
        gridspec_kw={'height_ratios': [1, 2], 'hspace': 0.05} 
    )
    
    metadata = {"Figure_Title": title, "Data_Summary": {}}

    # 2. TIME VECTOR GENERATION
    t_start = -epochs.get('Base', 3.0) 
    n_bins = speed_trial.shape[2]
    x_time = np.arange(n_bins) / fs + t_start
    #n_trials = speed_trial.shape[1]
    n_animals = speed_trial.shape[0]

    # 3. COLOR GRADIENT (Lightest to Darkest)
    base_color = colors[0] if isinstance(colors, list) else colors
    palette = sns.light_palette(base_color, n_colors=n_trials + 2)[2:]

    # 4. PLOT DYNAMICS ON BOTH AXES
    for t in range(n_trials):
        mean_curve = np.nanmean(speed_trial[:, t, :], axis=0)
        sem_curve = np.nanstd(speed_trial[:, t, :], axis=0) / np.sqrt(n_animals)
        
        c = palette[t]
        lw = 0.5 + (t / (n_trials - 1)) * 0.75 
        
        for ax in (ax_top, ax_bottom):
            ax.plot(x_time, mean_curve, color=c, lw=lw, label=f"T{t+1}", zorder=3)
            
            # Shade SEM for the First (Naive) and Last (Learned) trials only
            if t == 0 or t == (n_trials - 1):
                ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                                color=c, alpha=0.15, lw=0, zorder=2)

    # 5. EPOCH BACKGROUND SHADING
    epoch_shading_map = {
        'CS': ('#4091cf', 0.1),
        'Trace': ('gray', 0.05),
        'US': ('#e1703c', 0.15),
        'P_US1': ('gray', 0.05),
        'P_US2': ('gray', 0.1)
    }   
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue
            
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            for ax in (ax_top, ax_bottom):
                ax.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)
                
        current_time += ep_dur

    # 6. Y-AXIS LIMITS & BROKEN EFFECT
    max_speed = np.nanmax(np.nanmean(speed_trial, axis=0))
    top_limit = max(20.0, max_speed + 5.0) 
    
    ax_top.set_ylim(15, top_limit)
    ax_bottom.set_ylim(0, 5)

    # Hide the spines between the top and bottom axes
    ax_top.spines['bottom'].set_visible(False)
    ax_bottom.spines['top'].set_visible(False)
    ax_top.tick_params(labeltop=False, bottom=False)  
    ax_bottom.xaxis.tick_bottom()

    # Draw the diagonal hash marks (//)
    # 'd' is the physical length of the little slanted lines, NOT the gap between axes.
    d = .015  
    kwargs = dict(transform=ax_top.transAxes, color='k', clip_on=False, lw=0.75)
    ax_top.plot((-d, +d), (-d, +d), **kwargs)        
    
    kwargs.update(transform=ax_bottom.transAxes)  
    ax_bottom.plot((-d, +d), (1 - d, 1 + d), **kwargs)  

    # 7. FORMATTING
    ax_bottom.set_xlabel('Time from CS onset (s)', labelpad=1)
    
    # Removed the hardcoded x=0.02 so the layout engine can calculate the padding correctly
    #fig.supylabel(y_label, fontweight='bold') #
    ax_bottom.set_ylabel(y_label, labelpad=0.1)
    ax_bottom.yaxis.set_label_coords(-0.12, 0.8)

    for ax in (ax_top, ax_bottom):
        ax.set_xlim(t_start, x_time[-1])
        ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(0.5)
        ax.tick_params(axis='both', pad=1)
        
    ax_bottom.spines['bottom'].set_linewidth(0.5)
    
    # Ticks formatting
    ax_bottom.yaxis.set_major_locator(ticker.MultipleLocator(2))
    ax_top.yaxis.set_major_locator(ticker.MaxNLocator(nbins=3))
    
    # Compact legend
    ax_top.legend(frameon=False, ncol=2, loc='upper right', bbox_to_anchor=(1.05, 1.2),
                 handlelength=1.0, handletextpad=0.4, columnspacing=0.8) 

    # 8. EXPORT
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    
    fig.set_constrained_layout_pads(h_pad=0.0, hspace=0.0)
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()

## 1.1 PCA trajectory of all eligible cells 

In [ ]:
# 3D plotting with residuals
group_name = ['01.EC5b', '02.CA1-C', '03.CA1-I', '05.EC3-C', '06.EC3-I'] 
group_size = len(group_name) 

base_du = 20 
post_du = 3+6 
epochs = {
    'base':  (0,   int(base_du*fs)),
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du)*fs)) # Combining Post1 and Post2 for visual simplicity
    #'iti': (int((base_du+20+20+3+post_du)*fs), int((base_du+20+20+3+post_du+20)*fs))
}
epoch_durs = {'base':20, 'cs': 20, 'trace': 20, 'us': 3, 'pus': post_du}
epoch_colors = {'base': 'gray', 'cs': '#1f77b4', 'trace': '#2ca02c', 'us': '#d62728', 'pus': 'purple'}   

bins_plot_start = int((20-base_du)*fs) 
bins_plot_end = int(sum(epoch_durs.values())*fs) 

width_mm = 50  # 
# PLot with residuals
test_algori_data = 'post_02_2_resp_cal_new_z_residual'
plot_idx = 1
for i in range(group_size):    
    dpath_test = os.path.join(dpath_cal_all, test_algori_data)
    print(group_name[i])  
    with open(os.path.join(dpath_test, group_name[i] + '_ds_cal_cell_wise_union.pkl'), 'rb') as f:
            ds_data = pickle.load(f)  
    
    # All eligible cells   
    Sig_bin_trial = ds_data['whole'][:, :, bins_plot_start: bins_plot_end]
    Sig_bin_trial = Sig_bin_trial.transpose(1, 0, 2) # From (trials, unit_id, bins) To shape (unit_id, n_trials, n_bins)    

    if (group_name[i] == '02.CA1-C') | (group_name[i] == '03.CA1-I'):
        width_mm =50
    elif group_name[i] == '01.EC5b':
        width_mm =40
    else:
        width_mm = 45
    plot_pca_3d(Sig_bin_trial, [1,2,6], epochs, epoch_colors, fs, width_mm, dpath_plot, f'01_{plot_idx}_PCA trajectory_all_eligible_cells_residuals-{group_name[i]}')
    plot_idx +=1
     
print('All finished************')         

In [ ]:
# 2. plot with original calcium traces
group_name = ['02.CA1-C','03.CA1-I']
group_size = len(group_name) 
width_mm = 50  # 
# PLot with residuals
test_algori_data = 'post_02_2_resp_cal_new_z'
plot_idx = 1
for i in range(group_size):    
    dpath_test = os.path.join(dpath_cal_all, test_algori_data)
    print(group_name[i])  
    with open(os.path.join(dpath_test, group_name[i] + '_ds_cal_cell_wise_union.pkl'), 'rb') as f:
            ds_data = pickle.load(f)      
    # All eligible cells   
    Sig_bin_trial = ds_data['whole'][:, :, bins_plot_start: bins_plot_end]
    Sig_bin_trial = Sig_bin_trial.transpose(1, 0, 2) # From (trials, unit_id, bins) To shape (unit_id, n_trials, n_bins)    

    plot_pca_3d(Sig_bin_trial, [1,2,6], epochs, epoch_colors, fs, width_mm, dpath_plot, f'sup_01_{plot_idx}_PCA trajectory_all_eligible_cells_raw values-{group_name[i]}')
    plot_idx +=1
print('All finished************')    

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
from scipy.ndimage import gaussian_filter1d
from sklearn.decomposition import PCA

def plot_pca_3d(data_3d, trials_to_plot, epochs, epoch_colors, fs, width_mm, output_path, title, flag_bs_subtraction=0):
    """
    Plots 3D PCA trajectories for multiple learning stages (e.g., T1, T2, T6) in a single cubic panel.
    Applies visual hierarchy (alpha/linewidth/style) to prevent overlapping clutter.
    
    Parameters:
    -----------
    data_3d : np.array
        Shape (n_cells, n_trials, n_bins).
    trials_to_plot : list of int
        1-indexed trials to plot (e.g., [1, 2, 6] for Initial, Early, Late).
    width_mm : float
        Target width (height will match to ensure a cubic projection).
    """
    set_pub_style()
    n_cells, n_trials, n_bins = data_3d.shape   
    
    # 1. TRIAL-BY-TRIAL BASELINE SUBTRACTION
    if flag_bs_subtraction == 1:
        base_start, base_end = epochs['Base']
        for t in range(n_trials):
            trial_base_mean = np.mean(data_3d[:, t, base_start:base_end], axis=1, keepdims=True)
            data_3d[:, t, :] -= trial_base_mean
            
    # 2. TEMPORAL SMOOTHING
    data_smoothed = gaussian_filter1d(data_3d, sigma=2, axis=2)  
    
    # 3. RUN PCA ON CONCATENATED DATA
    data_concat = data_smoothed.transpose(1, 2, 0).reshape(n_trials * n_bins, n_cells)
    pca = PCA(n_components=3)
    pcs_concat = pca.fit_transform(data_concat)
    pcs = pcs_concat.reshape(n_trials, n_bins, 3)   
    var_exp = pca.explained_variance_ratio_ * 100
    
    metadata = {
        "Figure_Title": title,
        "Cell_Count": n_cells,
        "Explained_Variance": {
            "PC1": float(var_exp[0]),
            "PC2": float(var_exp[1]),
            "PC3": float(var_exp[2]),
            "Total_3D": float(np.sum(var_exp[:3]))
        },
        "Trials_Plotted": trials_to_plot
    }

    # 4. SET UP 3D PLOT (Enforcing a Square/Cubic aspect ratio)
    fig = plt.figure(figsize=(width_mm / 25.4, width_mm / 25.4), dpi=300, layout='constrained')
    ax = fig.add_subplot(111, projection='3d')
    
    style_hierarchy = [
        {'ls': '-',  'lw': 0.5, 'alpha': 1.0,  'marker': '*', 's': 20, 'label': 'Initial'},
        {'ls': '--', 'lw': 0.5, 'alpha': 1.0,  'marker': 'o', 's': 10, 'label': 'Early'},
        {'ls': '-',  'lw': 1.0, 'alpha': 0.25, 'marker': 'o', 's': 20, 'label': 'Late'}
    ]

    # 5. PLOT TRAJECTORIES
    starter_bin = int(17 * fs) # Only plot recent 3s base
    for i, t_num in enumerate(trials_to_plot):
        t_idx = t_num - 1 # 0-indexed
        t_pcs = pcs[t_idx]
        sty = style_hierarchy[min(i, len(style_hierarchy)-1)]
        
        # Plot Segments
        for name, (start, end) in epochs.items():
            if name == 'base':
                start = starter_bin 
            plot_end = min(end + 1, n_bins)
            
            ax.plot(t_pcs[start:plot_end, 0], t_pcs[start:plot_end, 1], t_pcs[start:plot_end, 2], 
                    color=epoch_colors[name], linestyle=sty['ls'], linewidth=sty['lw'], alpha=sty['alpha'])
            
        # Plot Start Marker
        ax.scatter(t_pcs[starter_bin, 0], t_pcs[starter_bin, 1], t_pcs[starter_bin, 2], 
                   color='yellow', edgecolor='black', marker=sty['marker'], 
                   s=sty['s'], linewidth=0.4, alpha=sty['alpha'], zorder=10)

    # 6. AXES FORMATTING (Low Ink Standard)
    # Restore the axis labels, keeping them clean and light
    try:
        ax.set_box_aspect(None, zoom=0.95)
    except TypeError:
        # Fallback for older versions of Matplotlib
        ax.dist = 12
    ax.tick_params(axis='x', pad=-5)
    ax.tick_params(axis='y', pad=-5)
    ax.tick_params(axis='z', pad=-4)

    ax.set_xlabel(f'PC1 ({var_exp[0]:.1f}%)', labelpad=-12) #({var_exp[0]:.1f}%) 
    ax.set_ylabel(f'PC2 ({var_exp[1]:.1f}%)', labelpad=-11) # the smaller , the closer to axis
    ax.set_zlabel(f'PC3 ({var_exp[2]:.1f}%)', labelpad=-26)   
    # ---------------------------------------------------------
    
    # 1. Remove the gray pane backgrounds
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.set_major_locator(ticker.MaxNLocator(6)) #MaxNLocator(nbins=4)
    ax.yaxis.set_major_locator(ticker.MaxNLocator(6)) 
    ax.zaxis.set_major_locator(ticker.MaxNLocator(6)) 
    #ax.xaxis.set_major_locator(ticker.MultipleLocator(1))

    
    # 2. Turn off the thick structural edges around the panes
    ax.xaxis.pane.set_edgecolor('none')
    ax.yaxis.pane.set_edgecolor('none')
    ax.zaxis.pane.set_edgecolor('none')

    # 3. Direct dictionary manipulation of the 3D grid properties
    # RGBA: (Red, Green, Blue, Alpha). 0.85 is a very light silver.
    micro_grid_color = (0.7, 0.7, 0.7, 0.6) 
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis._axinfo["grid"]['color'] = micro_grid_color
        axis._axinfo["grid"]['linewidth'] = 0.2
        axis._axinfo["grid"]['linestyle'] = '-'
        
    # ---------------------------------------------------------

    # 7. COMPACT LEGEND
    custom_lines = []
    # Trial structure
    for i, t_num in enumerate(trials_to_plot):
        sty = style_hierarchy[min(i, len(style_hierarchy)-1)]
        custom_lines.append(
            Line2D([0], [0], color='gray', linestyle=sty['ls'], lw=sty['lw'], alpha=sty['alpha'], 
                   marker=sty['marker'], markerfacecolor='yellow', markeredgecolor='black',
                   markersize=5 if sty['marker'] == '*' else 3, label=sty['label'])
        )
    
    # Spacer
    custom_lines.append(Line2D([], [], color='none', label='')) 
    
    # Epoch structure
    for name, color in epoch_colors.items():
        custom_lines.append(Line2D([0], [0], color=color, lw=1.5, label=name.capitalize()))
        
    #ax.legend(handles=custom_lines, loc='center left', bbox_to_anchor=(1.05, 0.5), 
    #          frameon=False, handletextpad=0.5, borderpad=0)

    # 8. EXPORT
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))       
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    
    plt.savefig(f"{base_path}.pdf", facecolor='white', edgecolor='none')
    plt.savefig(f"{base_path}.png", dpi=300, bbox_inches=None)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## supp_1.1 plotting of PC variances of all eligible cells 

In [ ]:
# 3D plotting with residuals
dict_variance = {'CA1': ['02.CA1-C', '03.CA1-I'], 
                 #'EC3': [ '05.EC3-C', '06.EC3-I'],
                 'EC3-C': [ '05.EC3-C'],
                 'EC3-I': [ '06.EC3-I'],
                 'EC5b': ['01.EC5b']
}

base_du = 20 # 10s
post_du = 3+6 #post shock  use 20s
epochs = {
    'base':  (0,   int(base_du*fs)),
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du)*fs)) # Combining Post1 and Post2 for visual simplicity
    #'iti': (int((base_du+20+20+3+post_du)*fs), int((base_du+20+20+3+post_du+20)*fs))
}
epoch_durs = {'base':20, 'cs': 20, 'trace': 20, 'us': 3, 'pus': post_du}
epoch_colors = {'base': 'gray', 'cs': '#1f77b4', 'trace': '#2ca02c', 'us': '#d62728', 'pus': 'purple'}   

bins_plot_start = int((20-base_du)*fs) # pre-shock 3s
bins_plot_end = int(sum(epoch_durs.values())*fs) 

height_mm = 25 
width_mm = 40  # 

# PLot with residuals
test_algori_data = 'post_02_2_resp_cal_new_z_residual'
for key, group_name in dict_variance.items():
    if key=='CA1':
        colors = ['#4091cf', '#e1703c'] # Blue, red, ,'#8cba54'
        group_keys = ['CA1-C', 'CA1-I'] #,  , 'CA1-E'
    #if key=='EC3':
    #    colors = ['#4091cf', '#e1703c']
    #    group_keys = ['EC3-C', 'EC3-I'] #, 
    if key=='EC3-C':
        colors = ['#4091cf']
        group_keys = ['EC3-C']
    if key=='EC3-I':
        colors = ['#e1703c']
        group_keys = ['EC3-I']
    if key=='EC5b':
        colors = ['#A6761D']
        group_keys = ['EC5b']    
    group_size = len(group_name)
    data_3d_group = {}
    for i in range(group_size):    
        dpath_test = os.path.join(dpath_cal_all, test_algori_data)
        print(group_name[i])  
        with open(os.path.join(dpath_test, group_name[i] + '_ds_cal_cell_wise_union.pkl'), 'rb') as f:
                ds_data = pickle.load(f)  
        
        # All eligible cells   
        Sig_bin_trial = ds_data['whole'][:, :, bins_plot_start: bins_plot_end]
        Sig_bin_trial = Sig_bin_trial.transpose(1, 0, 2) # From (trials, unit_id, bins) To shape (unit_id, n_trials, n_bins)    

        data_3d_group[group_keys[i]] = Sig_bin_trial
    # Plot the CDF of explained variances
    plot_PC_variances(data_3d_group, width_mm, height_mm, colors, fs, dpath_plot, f'sup_01_10_Explained variances of PCA_all_eligible_cells_residuals-{key}', var_threshold=90.0)

print('All finished************')         

In [ ]:
# PLot with original calcium traces
height_mm = 20 
width_mm = 40  # 

test_algori_data = 'post_02_2_resp_cal_new_z'
for key, group_name in dict_variance.items():
    if key=='CA1':
        colors = ['#4091cf', '#e1703c'] # Blue, red, '#8cba54'
        group_keys = ['CA1-C', 'CA1-I'] 
    if key=='EC3':
        colors = ['#4091cf', '#e1703c']
        group_keys = ['EC3-C', 'EC3-I'] 
        
    group_size = len(group_name)
    data_3d_group = {}
    for i in range(group_size):    
        dpath_test = os.path.join(dpath_cal_all, test_algori_data)
        print(group_name[i])  
        with open(os.path.join(dpath_test, group_name[i] + '_ds_cal_cell_wise_union.pkl'), 'rb') as f:
                ds_data = pickle.load(f)  
        
        # All eligible cells   
        Sig_bin_trial = ds_data['whole'][:, :, bins_plot_start: bins_plot_end]
        Sig_bin_trial = Sig_bin_trial.transpose(1, 0, 2) # From (trials, unit_id, bins) To shape (unit_id, n_trials, n_bins)    

        data_3d_group[group_keys[i]] = Sig_bin_trial
    # Plot the CDF of explained variances
    plot_PC_variances(data_3d_group, width_mm, height_mm, colors, fs, dpath_plot, f'sup_01_11_Explained variances of PCA_all_eligible_cells_raw data-{key}', var_threshold=90.0)

print('All finished************')         

In [ ]:
def plot_PC_variances(data_3d_group, width_mm, height_mm, colors, fs, output_path, title, var_threshold=90.0):
    """
    Plots the Cumulative Explained Variance for multiple groups.
    Dynamically truncates the X-axis to the minimum number of PCs required 
    to reach the target variance threshold (default 90%) across all groups.
    
    Parameters:
    -----------
    data_3d_group : dict
        Dictionary of 3D arrays (n_cells, n_trials, n_bins) per group.
    var_threshold : float
        The cumulative variance percentage to highlight and truncate the axis by (e.g., 90.0).
    """
    set_pub_style()
    group_keys = list(data_3d_group.keys())
    
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    metadata = {
        "Figure_Title": title,
        "Variance_Threshold_Target": var_threshold,
        "Groups": {}
    }
    
    max_pcs_to_threshold = 0
    group_curves = []

    # --- 1. COMPUTE PCA & EXTRACT VARIANCE CURVES ---
    for idx, g_key in enumerate(group_keys):
        data_3d = data_3d_group[g_key]
        n_cells, n_trials, n_bins = data_3d.shape
        
        # Temporal Smoothing
        data_smoothed = gaussian_filter1d(data_3d, sigma=2, axis=2)  
        
        # Flatten for PCA
        data_concat = data_smoothed.transpose(1, 2, 0).reshape(n_trials * n_bins, n_cells)
        
        # Fit PCA to extract variance for all available components
        pca = PCA()
        pca.fit(data_concat)
        
        # Calculate cumulative variance (%)
        cum_var = np.cumsum(pca.explained_variance_ratio_) * 100
        
        # Find exactly how many PCs are needed to reach the threshold
        if cum_var[-1] >= var_threshold:
            pc_thresh = np.argmax(cum_var >= var_threshold) + 1
        else:
            pc_thresh = len(cum_var)
            
        max_pcs_to_threshold = max(max_pcs_to_threshold, pc_thresh)
        group_curves.append(cum_var)
        
        # Log precise metadata
        metadata["Groups"][g_key] = {
            "Cell_Count": n_cells,
            "Total_PCs": len(cum_var),
            f"PCs_Needed_For_{var_threshold}%": int(pc_thresh),
            "Variance_PC1": float(cum_var[0]),
            "Variance_PC1_to_PC3": float(cum_var[2]) if len(cum_var) >= 3 else float(cum_var[-1])
        }

    # --- 2. PLOTTING THE CURVES ---
    for idx, (g_key, cum_var) in enumerate(zip(group_keys, group_curves)):
        pcs = np.arange(1, len(cum_var) + 1)
        ax.plot(pcs, cum_var, color=colors[idx], lw=1.0, label=g_key, zorder=3)
        
    # --- 3. AESTHETICS & THRESHOLD MARKERS ---
    # Draw a subtle horizontal line at the 90% threshold
    ax.axhline(var_threshold, color='gray', linestyle=':', lw=0.75, alpha=0.6, zorder=1)

    # Formatting Typography (Low ink, no bold)
    ax.set_xlabel('Principal Component', labelpad=0.1)
    ax.set_ylabel('Cumulative\nVariance (%)', labelpad=0.1)
    
    # Apply Log Scale to X-axis
    ax.set_xscale('log')
    
    # Scale X-axis to exactly frame the area where groups cross the 90% threshold
    x_max_plot = int(max_pcs_to_threshold * 1.1) + 1 # Add 10% breathing room
    ax.set_xlim(1, x_max_plot)
    ax.set_ylim(0, 105) # Cap slightly above 100%
    
    # Clean tick marks for Log Scale
    ax.xaxis.set_major_locator(ticker.LogLocator(base=10.0))
    # ScalarFormatter prevents the axis from displaying scientific notation (e.g., 10^1)
    ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
    
    ax.yaxis.set_major_locator(ticker.MultipleLocator(20))
    
    ax.tick_params(axis='both', length=2, width=0.5, pad=0.5)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    
    # Minimalist Custom Legend
    custom_lines = [Line2D([0], [0], color=colors[i], lw=1.0, label=g_key) 
                    for i, g_key in enumerate(group_keys)]
    
    ax.legend(handles=custom_lines, frameon=False, loc='lower right', handlelength=1.5, handletextpad=0.4, borderpad=0.1)

    # --- 4. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))       
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 1.2 Mean z scores of the residual after GLM

In [ ]:
group_name = ['02.CA1-C']  #'02.CA1-C'

group_size = len(group_name)
test_algori_data_raw = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'
test_algori_data_residual = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z_residual' 

base_du = 3 
post_du = 40 
bins_trial_start = int((20-base_du)*fs) 
bins_trial_end = int((20+20+20+3+ post_du)*fs) 

cal_mean_2_trials_animal = dict()

states = ['original', 'residual']
group_keys = ['Original', 'Residual'] 

for i, state in enumerate(states):
    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[0])
    if state=='original':
        dpath_test = os.path.join(dpath_cal_group, test_algori_data_raw)
    else:
        dpath_test = os.path.join(dpath_cal_group, test_algori_data_residual)
    print(dpath_test) 

    Sig_bin_trial = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials.nc"))        
    Sig_bin_trial = Sig_bin_trial['Sig_each_trial'].sel(session='session0_condi')           
    #Extract data
    cal_mean_2_trials_animal[group_keys[i]] = Sig_bin_trial.sel(trials=range(0,2), bins=range(bins_trial_start, bins_trial_end)).mean(dim='trials').values
  
width_mm = 40  # 
height_mm = 30 #  
epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'P_US1': 3, 'P_US2': 6}

colors = ['black', '#0072B2'] # deep blue

# 2. plot of first 2 trials -Mean Z score 
plot_trial_population_activity(cal_mean_2_trials_animal,'Mean Z-Score', width_mm, height_mm, epochs, group_keys, colors, fs, dpath_plot, '01_0_2_First 2 trials-mean Z score of residual-CA1-animal-wise')
#plot_trial_population_activity(cal_mean_2_trials_animal,'Mean Z-Score', width_mm, height_mm, epochs, group_keys, colors, fs, dpath_plot, '01_0_2_First 2 trials-mean Z score of residual-EC3-animal-wise')
print('All finished************') 

In [ ]:
def plot_trial_population_activity(cal_trial, y_label, width_mm, height_mm, epochs, group_keys, colors, fs, output_path, title):
    """
    Plots the Trial-Averaged Population Activity for the entire TFC trial.
    Locked to strict millimeter layout (e.g., 60x30mm).
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    metadata = {"Figure_Title": title, "Data_Summary": {}}

    # --- 1. TIME VECTOR GENERATION ---
    # 0 is Tone Onset. Baseline represents negative time.
    t_start = -epochs.get('Base', 3.0) 
    
    # Extract number of bins from the first available dataset
    first_key = [k for k in group_keys if k in cal_trial][0]
    n_bins = cal_trial[first_key].shape[1]
    
    # Create time vector in SECONDS using the sampling frequency (fs)
    x_time = np.arange(n_bins) / fs + t_start

    # --- 2. DYNAMIC EPOCH SHADING ---
    epoch_shading_map = {
        'CS': ('#4091cf', 0.1),       # Tone: Blue
        'Trace': ('gray', 0.05),      # Trace: Light Gray
        'US': ('#e1703c', 0.15),      # Shock: Orange
        'P_US1': ('gray', 0.05),      # Post1: Light Gray
        'P_US2': ('gray', 0.1)}       # Postw: dark Gray        
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur

    # --- 3. PLOT MEAN + SEM CURVES ---
    for i, grp in enumerate(group_keys):
        if grp not in cal_trial: continue
        
        data = cal_trial[grp]
        n_cells = data.shape[0]
        
        mean_curve = np.nanmean(data, axis=0)
        sem_curve = np.nanstd(data, axis=0) / np.sqrt(n_cells)
        color = colors[i]        
        # Micro-thin lines (0.75) to prevent ink crowding
        if (y_label=='Mean Speed (cm/s)') & (i==1):
            ax.plot(x_time, mean_curve, color=color, lw=0.75, label=f"{grp})", linestyle='--', zorder=3) # (n={n_cells}
        else:
            ax.plot(x_time, mean_curve, color=color, lw=0.75, label=f"{grp}", zorder=3)
            
        ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                        color=color, alpha=0.3, lw=0, zorder=2)

        metadata["Data_Summary"][grp] = {"N_mice": n_cells, "Peak_Z": float(np.nanmax(mean_curve))}

    # --- 4. FORMATTING ---
    ax.set_xlabel('Time from CS onset (s)',  labelpad=1)
    ax.set_ylabel(y_label, labelpad=0.1)
    
    # Strict limits based on exact epoch timings
    ax.set_xlim(t_start, x_time[-1])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    if y_label == 'Mean Z-Score':
        ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    else:
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    #ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=6))
    # length=2 to ensure the physical tick marks stay tiny.
    ax.tick_params(axis='both') #, length=2, pad=1
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.legend(loc='upper left', frameon=False, handlelength=1.5)
    # --- 5. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 2.1~2.3 Statistics of PCA-Macro-Sequence with all cells

In [ ]:
group_name = ['01.EC5b', '02.CA1-C', '03.CA1-I', '05.EC3-C', '06.EC3-I']
group_size = len(group_name) 

width_mm = 40  # 
height_mm = 30 #  

test_algori_data = 'post_02_2_resp_cal_new_z_residual'

base_du = 20     
stat_base_du = 3  # The strict statistical baseline immediately preceding CS
post_du1 = 3
post_du2 = 6
epochs = {
    'context': (0, int((base_du - stat_base_du) * fs)), 
    # 17s to 20s: The strict 3s origin for statistics
    'base':    (int((base_du - stat_base_du) * fs), int(base_du * fs)), 
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus_1':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du1)*fs)), # Combining Post1 and Post2 for visual simplicity
    'pus_2':  (int((base_du+20+20+3+post_du1)*fs), int((base_du+20+20+3+post_du1+post_du2)*fs))
}

epoch_durs = {'base':20, 'cs': 20, 'trace': 20, 'us': 3, 'pus_1': post_du1, 'pus_2': post_du2}

bins_start = int((20-base_du)*fs) # pre-shock 3s
bins_end = int(sum(epoch_durs.values())*fs) # 
colors = ['#dfc27d', '#bf812d', '#543005'] #Light Bronze, Medium Copper, Deep Brown/Dark Copper
for i in range(group_size):       
    dpath_test = os.path.join(dpath_cal_all, test_algori_data)
    print(group_name[i])  
    with open(os.path.join(dpath_test, f'{group_name[i]}_ds_cal_animal_wise.pkl'), 'rb') as f:
            ds_data = pickle.load(f)  
    
    # All eligible cells   
    Sig_bin_trial = []
    for animal_i in range(len(ds_data['whole'])):
        #if ds_data['whole'][animal_i].shape[1] > 300:
        Sig_bin_trial.append(ds_data['whole'][animal_i][:, :, bins_start: bins_end])
      
    time_from_cs, stats_dict = extract_macro_pca_stat_animal_wise(Sig_bin_trial, epochs, fs=5.0, flag_smoothing=True, metric_sigma=2.0)
    height_mm = 25 # 
    plot_pca_distance_animal_wise(time_from_cs, stats_dict, epochs, fs, width_mm, height_mm, colors,  group_name[i], dpath_plot, f"02_1_PCA_Distance_Animal_Wise_{group_name[i]}")
    height_mm = 25
    plot_pca_velocity_animal_wise(time_from_cs, stats_dict, epochs, fs, width_mm, height_mm, colors,  group_name[i], dpath_plot, f"02_2_PCA_Velocity_Animal_Wise_{group_name[i]}")
    plot_pca_tortuosity_animal_wise(stats_dict, epochs, fs, width_mm, height_mm, colors, dpath_plot, f"02_3_PCA_Tortuosity_Animal_Wise_{group_name[i]}")

print('All finished************')         

In [ ]:
def plot_pca_distance_animal_wise(time_vec, stats_dict, epochs, fs, width_mm, height_mm, colors, group_name, output_path, title="01_PCA_Distance"):
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    data = stats_dict['distance']
    n_animals, n_trials, _ = data.shape
    trials_to_plot = [0, 1, n_trials - 1]
    #labels = ['T1', 'T2', f'T{n_trials}']
    labels = ['T1 (Initial)', 'T2 (Early)', f'T{n_trials} (Late)']
    metadata = {"Figure_Title": title, "Trials_Plotted": {}}
    base_start_bin = epochs['base'][0]
    base_end_bin = epochs['base'][1]
    _apply_epoch_shading(ax, epochs, fs, base_end_bin)

    for idx, t_idx in enumerate(trials_to_plot):
        t_data = data[:, t_idx, base_start_bin:] # Crop plot to strictly Base->End
        valid_data = t_data[~np.isnan(t_data).any(axis=1)]
        n_valid = len(valid_data)
        if n_valid == 0: continue
            
        m = np.mean(valid_data, axis=0)
        s = np.std(valid_data, axis=0) / np.sqrt(n_valid)
        
        plot_time = time_vec[base_start_bin:]
        ax.plot(plot_time, m, color=colors[idx], lw=0.75, label=labels[idx], zorder=3)
        if idx==0:
            ax.fill_between(plot_time, m - s, m + s, color=colors[idx], alpha=0.3, lw=0, zorder=2)
        else:
            ax.fill_between(plot_time, m - s, m + s, color=colors[idx], alpha=0.1, lw=0, zorder=2)
        metadata["Trials_Plotted"][f"Trial_{t_idx+1}"] = {"N_animals": n_valid, "Max_Dist": float(np.max(m))}
    if group_name == '02.CA1-C':
        ax.set_xlabel('Time from CS onset (s)', labelpad=1, color='none')
    else:
        ax.set_xlabel('Time from CS onset (s)', labelpad=1)
    ax.set_ylabel('State dist. to base', labelpad=0.1)
    
    # Start plot at -3s
    t_start = (base_start_bin - base_end_bin) / fs
    ax.set_xlim(t_start, time_vec[-1])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    
    ax.tick_params(axis='both', length=2, pad=1)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_linewidth(0.5)
    ax.legend(loc='best', frameon=False, handlelength=1.5)

    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    save_metadata_json(metadata, output_path, title)


def plot_pca_velocity_animal_wise(time_vec, stats_dict, epochs, fs, width_mm, height_mm, colors, group_name, output_path, title="02_PCA_Velocity"):
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    data = stats_dict['velocity']
    n_animals, n_trials, _ = data.shape
    trials_to_plot = [0, 1, n_trials - 1]
    labels = ['T1 (Initial)', 'T2 (Early)', f'T{n_trials} (Late)']

    metadata = {"Figure_Title": title, "Trials_Plotted": {}}
    base_start_bin = epochs['base'][0]
    base_end_bin = epochs['base'][1]
    _apply_epoch_shading(ax, epochs, fs, base_end_bin)

    for idx, t_idx in enumerate(trials_to_plot):
        t_data = data[:, t_idx, base_start_bin:] # Crop plot
        valid_data = t_data[~np.isnan(t_data).any(axis=1)]
        n_valid = len(valid_data)
        if n_valid == 0: continue
            
        m = np.mean(valid_data, axis=0)
        s = np.std(valid_data, axis=0) / np.sqrt(n_valid)
        
        plot_time = time_vec[base_start_bin:]
        ax.plot(plot_time, m, color=colors[idx], lw=0.75, label=labels[idx], zorder=3)
        if idx==0:
            ax.fill_between(plot_time, m - s, m + s, color=colors[idx], alpha=0.3, lw=0, zorder=2)
        else:
            ax.fill_between(plot_time, m - s, m + s, color=colors[idx], alpha=0.1, lw=0, zorder=2)
            
        metadata["Trials_Plotted"][f"Trial_{t_idx+1}"] = {"N_animals": n_valid, "Max_Velo": float(np.max(m))}

    if group_name == '02.CA1-C':
        ax.set_xlabel('Time from CS onset (s)', labelpad=1, color='none')
    else:
        ax.set_xlabel('Time from CS onset (s)', labelpad=1)
    ax.set_ylabel('Trajectory velocity', labelpad=0.1)
    
    t_start = (base_start_bin - base_end_bin) / fs
    ax.set_xlim(t_start, time_vec[-1])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    
    ax.tick_params(axis='both', length=2, pad=1)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_linewidth(0.5)
    #ax.legend(loc='best', frameon=False, handlelength=1.5)

    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    save_metadata_json(metadata, output_path, title)


def plot_pca_tortuosity_animal_wise(stats_dict, epochs, fs, width_mm, height_mm, colors, output_path, title="03_PCA_Tortuosity"):
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    data = stats_dict['tortuosity']
    chunk_times = stats_dict['tort_times']
    
    n_animals, n_trials, _ = data.shape
    trials_to_plot = [0, 1, n_trials - 1]
    labels = ['T1 (Initial)', 'T2 (Early)', f'T{n_trials} (Late)']

    metadata = {"Figure_Title": title, "Trials_Plotted": {}}
    base_end_bin = epochs['base'][1]
    _apply_epoch_shading(ax, epochs, fs, base_end_bin)

    for idx, t_idx in enumerate(trials_to_plot):
        t_data = data[:, t_idx, :]
        valid_data = t_data[~np.isnan(t_data).any(axis=1)]
        n_valid = len(valid_data)
        if n_valid == 0: continue
            
        m = np.mean(valid_data, axis=0)
        s = np.std(valid_data, axis=0) / np.sqrt(n_valid)
        
        ax.plot(chunk_times, m, color=colors[idx], lw=0.75, marker='o', markersize=2.5, label=labels[idx], zorder=3)
        ax.errorbar(chunk_times, m, yerr=s, fmt='none', color=colors[idx], elinewidth=0.75, capsize=0, zorder=2)
        metadata["Trials_Plotted"][f"Trial_{t_idx+1}"] = {"N_animals": n_valid}

    ax.set_xlabel('Time from CS onset (s)', labelpad=1)
    ax.set_ylabel('Epoch tortuosity', labelpad=0.1)
    
    t_span = chunk_times[-1] - chunk_times[0]
    ax.set_xlim(chunk_times[0] - (t_span * 0.05), chunk_times[-1] + (t_span * 0.05))
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    
    ax.tick_params(axis='both', length=2, pad=1)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_linewidth(0.5)
    #ax.legend(loc='upper right', frameon=False, handlelength=1.5)

    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    save_metadata_json(metadata, output_path, title)
        
def extract_macro_pca_stat_animal_wise(data_3d_animals, epochs, fs=5.0, flag_smoothing=True, metric_sigma=2.0):
    """
    Extracts Euclidean distance, step velocity, and chunked tortuosity on an ANIMAL-WISE basis.
    Uses 'context' (e.g., 0-17s) for PCA fitting context, but anchors distance origin to strictly 'base' (e.g., 17-20s).
    """
    n_animals = len(data_3d_animals)
    first_valid_animal = next(a for a in data_3d_animals if a.size > 0)
    n_trials, _, n_bins = first_valid_animal.shape
    
    # 1. Define Temporal Alignments
    base_start = epochs.get('base')[0]
    base_end = epochs.get('base')[1]
    
    # Time vector for continuous metrics (spanning context to end), 0s = CS Onset
    time_vec = (np.arange(n_bins) - base_end) / fs
    
    # Define explicit chunks for Tortuosity (ignoring Context to save space)
    chunks = []
    chunk_times = []
    for ep_name, (start, end) in epochs.items():
        ep_name_lower = ep_name.lower()
        if ep_name_lower == 'context': 
            continue
            
        if 'trace' in ep_name_lower and (end - start) / fs >= 20:
            mid = start + (end - start) // 2
            chunks.append((start, mid))
            chunk_times.append((start + mid) / 2 / fs - base_end / fs)
            
            chunks.append((mid, end))
            chunk_times.append((mid + end) / 2 / fs - base_end / fs)
        else:
            chunks.append((start, end))
            chunk_times.append((start + end) / 2 / fs - base_end / fs)
            
    n_chunks = len(chunks)
    
    dist_all = np.full((n_animals, n_trials, n_bins), np.nan)
    velo_all = np.full((n_animals, n_trials, n_bins), np.nan)
    tort_all = np.full((n_animals, n_trials, n_chunks), np.nan)
    
    for a_idx, animal_data in enumerate(data_3d_animals):
        if animal_data.size == 0 or np.isnan(animal_data).all(): continue
        n_t, n_cells, n_b = animal_data.shape
        if n_cells < 3: continue
            
        data_smoothed = gaussian_filter1d(animal_data, sigma=2, axis=2)
        concat_data = np.transpose(data_smoothed, (1, 0, 2)).reshape(n_cells, -1)
        
        pca = PCA(n_components=3)
        pca_trials = pca.fit_transform(concat_data.T).reshape(n_t, n_b, 3)
        
        for t in range(n_t):
            # -- Euclidean Distance (Referenced STRICTLY to Base Centroid) --
            base_origin = np.mean(pca_trials[t, base_start:base_end, :], axis=0)
            raw_dist = np.linalg.norm(pca_trials[t, :, :] - base_origin, axis=1)
            
            # -- Velocity --
            step_diffs = np.diff(pca_trials[t, :, :], axis=0)
            raw_velo = np.linalg.norm(step_diffs, axis=1)
            raw_velo = np.insert(raw_velo, 0, 0) 
            
            if flag_smoothing:
                dist_all[a_idx, t, :] = gaussian_filter1d(raw_dist, sigma=metric_sigma)
                velo_all[a_idx, t, :] = gaussian_filter1d(raw_velo, sigma=metric_sigma)
            else:
                dist_all[a_idx, t, :] = raw_dist
                velo_all[a_idx, t, :] = raw_velo
                
            # -- Discrete Chunked Tortuosity --
            for i, (c_start, c_end) in enumerate(chunks):
                if c_end - c_start < 2:
                    tort_all[a_idx, t, i] = np.nan
                    continue
                    
                path_len = np.sum(raw_velo[c_start:c_end])
                dir_dist = np.linalg.norm(pca_trials[t, c_end-1, :] - pca_trials[t, c_start, :])
                
                if path_len < 1e-2:
                    tort_all[a_idx, t, i] = 1.0
                else:
                    raw_tort = path_len / (dir_dist + 1e-8)
                    tort_all[a_idx, t, i] = min(raw_tort, 20.0) 
                    
    stats_dict = {
        'distance': dist_all,
        'velocity': velo_all,
        'tortuosity': tort_all,
        'tort_times': np.array(chunk_times)
    }
    return time_vec, stats_dict

def _apply_epoch_shading(ax, epochs, fs, base_end_bin):
    """Helper to apply publication-style epoch shading to the background."""
    epoch_shading_map = {
        'cs': ('#4091cf', 0.1),       
        'trace': ('gray', 0.05),      
        'us': ('#e1703c', 0.15),      
        'pus_1': ('gray', 0.05),      
        'pus_2': ('gray', 0.15)
    }
    
    for ep_name, (start_bin, end_bin) in epochs.items():
        ep_name_lower = ep_name.lower()
        if 'base' in ep_name_lower or 'context' in ep_name_lower: 
            continue
            
        ep_start_sec = start_bin / fs - base_end_bin / fs
        ep_dur_sec = (end_bin - start_bin) / fs
        
        # Use strict dictionary matching to prevent 'us' from matching 'pus_1'
        if ep_name_lower in epoch_shading_map:
            c, a = epoch_shading_map[ep_name_lower]
            ax.axvspan(ep_start_sec, ep_start_sec + ep_dur_sec, color=c, alpha=a, lw=0, zorder=0) 

## 2.4~2.5 Statistics of State transition in PCA trajectory with all cells

In [ ]:
group_name = ['01.EC5b', '02.CA1-C', '03.CA1-I', '05.EC3-C', '06.EC3-I'] 
group_size = len(group_name) # 3

test_algori_data = 'post_02_2_resp_cal_new_z_residual'

base_du = 20      # Total pre-CS time
stat_base_du = 3  # The strict statistical baseline immediately preceding CS
post_du1 = 3
post_du2 = 6
epochs = {
    'context': (0, int((base_du - stat_base_du) * fs)), 
    # 17s to 20s: The strict 3s origin for statistics
    'base':    (int((base_du - stat_base_du) * fs), int(base_du * fs)), 
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus_1':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du1)*fs)), # Combining Post1 and Post2 for visual simplicity
    'pus_2':  (int((base_du+20+20+3+post_du1)*fs), int((base_du+20+20+3+post_du1+post_du2)*fs))
}

epoch_durs = {'base':20, 'cs': 20, 'trace': 20, 'us': 3, 'pus_1': post_du1, 'pus_2': post_du2}

bins_start = int((20-base_du)*fs) 
bins_end = int(sum(epoch_durs.values())*fs) 

height_mm = 40 #  
colors = ['#A6761D', '#845ec2', '#97cebf']
for i in range(group_size):       
    dpath_test = os.path.join(dpath_cal_all, test_algori_data)
    print(group_name[i])  
    with open(os.path.join(dpath_test, f'{group_name[i]}_ds_cal_animal_wise.pkl'), 'rb') as f:
            ds_data = pickle.load(f)  
    
    # All eligible cells   
    Sig_bin_trial = []
    for animal_i in range(len(ds_data['whole'])):
        Sig_bin_trial.append(ds_data['whole'][animal_i][:, :, bins_start: bins_end])

    matrix_stats = quantify_epoch_transitions_animal_wise(Sig_bin_trial, epochs, fs=5.0)
    # For Euclidean distance
    width_mm = 85  # 
    plot_epoch_distance_heatmap(matrix_stats, width_mm, height_mm, dpath_plot, f"02_4_Epoch_Distance_PCA_Animal_Wise_{group_name[i]}")
    # For consine similarity
    if group_name[i] == '02.CA1-C':
        width_mm =75
    else:
        width_mm = 85
    plot_epoch_cosine_heatmap(matrix_stats, width_mm, height_mm, dpath_plot, f"02_5_Epoch_Consine_similarity_PCA_Animal_Wise_{group_name[i]}")
print('All finished************')         

In [ ]:
def plot_epoch_distance_heatmap(matrix_stats, width_mm=70, height_mm=40, output_path="./", title="Epoch_Distance_Combined"):
    """
    Plots a 1x3 grid of 40mm-high heatmaps for Euclidean Distance (T1, T2, T6).
    Shared Y-axis, perfectly square cells, and unified minimalist colorbar.
    """
    set_pub_style()
    dist_all = matrix_stats['distance']
    epoch_names = matrix_stats['epoch_names']
    
    n_animals, n_trials, _, _ = dist_all.shape
    trials_to_plot = [0, 1, n_trials - 1]
    labels = ['T1 (Initial)', 'T2 (Early)', f'T{n_trials} (Late)']
    
    # Calculate global max across the plotted trials to lock the colorbar
    valid_dist = dist_all[:, trials_to_plot, :, :]
    global_max = np.nanmax(np.nanmean(valid_dist, axis=0))
    
    fig, axes = plt.subplots(1, 3, figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    metadata = {"Figure_Title": title, "Trials_Plotted": {}}
    
    for idx, (t_idx, ax) in enumerate(zip(trials_to_plot, axes)):
        t_data = dist_all[:, t_idx, :, :]
        valid_mask = ~np.isnan(t_data).any(axis=(1, 2))
        m_dist = np.nanmean(t_data[valid_mask], axis=0)
        n_valid = np.sum(valid_mask)
        
        metadata["Trials_Plotted"][labels[idx]] = int(n_valid)
        
        if n_valid > 0:
            sns.heatmap(m_dist, annot=True, fmt=".1f", cmap="Blues",
                        xticklabels=epoch_names, 
                        yticklabels=epoch_names if idx == 0 else False, 
                        vmin=0, vmax=global_max, ax=ax,
                        square=True,          
                        cbar=(idx == 2),      
                        annot_kws={"size": 5},
                        # --- ULTRA COMPACT COLORBAR SETTINGS ---
                        cbar_kws={
                            'label': 'Euclidean Distance', 
                            'shrink': 0.5,    # Reduces height to 50% of the plot
                            'aspect': 30,     # Higher number = thinner bar
                            'pad': 0.03       # Pulls it tightly against the right plot
                        })
            
            # Format the shared Colorbar text/edge
            if idx == 2:
                cbar = ax.collections[0].colorbar
                cbar.ax.tick_params(labelsize=4, length=1, pad=1)
                cbar.set_label('Euclidean Distance', size=5, labelpad=1)
                cbar.outline.set_linewidth(0.25)
                cbar.outline.set_edgecolor('gray')
            
        ax.set_title(labels[idx], fontsize=7, pad=2)
        ax.tick_params(axis='x', rotation=45, labelsize=5, pad=1, length=1)
        
        if idx == 0:
            ax.tick_params(axis='y', rotation=0, labelsize=5, pad=1, length=1)
        else:
            ax.tick_params(axis='y', length=0) # Hide tick marks on shared axes
            
        # Ensure no bold borders around the heatmaps
        for spine in ax.spines.values():
            spine.set_visible(False)

    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    save_metadata_json(metadata, output_path, title)

def plot_epoch_cosine_heatmap(matrix_stats, width_mm=70, height_mm=40, output_path="./", title="Epoch_Cosine_Combined"):
    """
    Plots a 1x3 grid of 40mm-high heatmaps for Cosine Similarity (T1, T2, T6).
    Shared Y-axis, perfectly square cells, and unified minimalist colorbar.
    """
    set_pub_style()
    cos_all = matrix_stats['cosine']
    active_epochs = matrix_stats['active_epochs']
    
    n_animals, n_trials, _, _ = cos_all.shape
    trials_to_plot = [0, 1, n_trials - 1]
    labels = ['T1 (Initial)', 'T2 (Early)', f'T{n_trials} (Late)']
    
    fig, axes = plt.subplots(1, 3, figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    metadata = {"Figure_Title": title, "Trials_Plotted": {}}
    
    for idx, (t_idx, ax) in enumerate(zip(trials_to_plot, axes)):
        t_data = cos_all[:, t_idx, :, :]
        valid_mask = ~np.isnan(t_data).any(axis=(1, 2))
        m_cos = np.nanmean(t_data[valid_mask], axis=0)
        n_valid = np.sum(valid_mask)
        
        metadata["Trials_Plotted"][labels[idx]] = int(n_valid)              
        if n_valid > 0:
            sns.heatmap(m_cos, annot=True, fmt=".1f", cmap="coolwarm",
                        xticklabels=active_epochs, 
                        yticklabels=active_epochs if idx == 0 else False,
                        vmin=-1.0, vmax=1.0, center=0, ax=ax,
                        square=True, 
                        cbar=(idx == 2), 
                        annot_kws={"size": 5},
                        cbar_kws={
                            'label': 'Cosine Similarity', 
                            'shrink': 0.5,    # Reduces height to 50% of the plot
                            'aspect': 30,     # Higher number = thinner bar
                            'pad': 0.03       # Pulls it tightly against the right plot
                        })
            
            if idx == 2:
                cbar = ax.collections[0].colorbar
                cbar.ax.tick_params(labelsize=4, length=1, pad=1)
                cbar.set_label('Cosine Similarity', size=5, labelpad=1)
                cbar.outline.set_linewidth(0.25)
                cbar.outline.set_edgecolor('gray')
            
        ax.set_title(labels[idx], fontsize=7, pad=2)
        ax.tick_params(axis='x', rotation=45, labelsize=5, pad=1, length=1)
        
        if idx == 0:
            ax.tick_params(axis='y', rotation=0, labelsize=5, pad=1, length=1)
        else:
            ax.tick_params(axis='y', length=0)
            
        for spine in ax.spines.values():
            spine.set_visible(False)

    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    save_metadata_json(metadata, output_path, title)
    
def quantify_epoch_transitions_animal_wise(data_3d_animals, epochs, fs=5.0):
    """
    Fits animal-wise PCA and calculates spatial distance and Cosine Similarity 
    between epoch centroids across trials.
    """
    n_animals = len(data_3d_animals)
    first_valid = next(a for a in data_3d_animals if a.size > 0)
    n_trials, _, n_bins = first_valid.shape
    
    # Filter out 'context' as it is only for PCA fitting, not statistics
    epoch_names = [ep for ep in epochs.keys() if ep.lower() != 'context']
    active_epochs = [ep for ep in epoch_names if ep.lower() != 'base']
    
    # Storage arrays: (n_animals, n_trials, ep, ep)
    dist_all = np.full((n_animals, n_trials, len(epoch_names), len(epoch_names)), np.nan)
    cos_all = np.full((n_animals, n_trials, len(active_epochs), len(active_epochs)), np.nan)
    
    for a_idx, animal_data in enumerate(data_3d_animals):
        if animal_data.size == 0 or np.isnan(animal_data).all(): 
            continue
            
        n_t, n_cells, n_b = animal_data.shape
        if n_cells < 3: 
            continue
            
        # Smooth and fit independent PCA for this animal using all bins
        data_smoothed = gaussian_filter1d(animal_data, sigma=2, axis=2)
        concat_data = np.transpose(data_smoothed, (1, 0, 2)).reshape(n_cells, -1)
        
        pca = PCA(n_components=3)
        pca_trials = pca.fit_transform(concat_data.T).reshape(n_t, n_b, 3)
        
        for t in range(n_t):
            centroids = {}
            for ep_name in epoch_names:
                start, end = epochs[ep_name]
                centroids[ep_name] = np.mean(pca_trials[t, start:end, :], axis=0)
                
            # 1. Distance Matrix (Includes Base)
            for i, ep1 in enumerate(epoch_names):
                for j, ep2 in enumerate(epoch_names):
                    dist_all[a_idx, t, i, j] = np.linalg.norm(centroids[ep1] - centroids[ep2])
                    
            # 2. Cosine Similarity (Relative to Base Origin)
            base_c = centroids[next(ep for ep in epoch_names if ep.lower() == 'base')]
            for i, ep1 in enumerate(active_epochs):
                v1 = centroids[ep1] - base_c
                for j, ep2 in enumerate(active_epochs):
                    v2 = centroids[ep2] - base_c
                    
                    norm_prod = np.linalg.norm(v1) * np.linalg.norm(v2)
                    if norm_prod < 1e-8:
                        cos_sim = 0.0
                    else:
                        cos_sim = np.dot(v1, v2) / norm_prod
                        
                    cos_all[a_idx, t, i, j] = np.clip(cos_sim, -1.0, 1.0)
                    
    matrix_stats = {
        'distance': dist_all,
        'cosine': cos_all,
        'epoch_names': epoch_names,
        'active_epochs': active_epochs
    }
    return matrix_stats

## 3.0 Distance to T6 trajectory demonstration with vector

In [ ]:
# 3D plotting with residuals
group_name = ['05.EC3-C'] 
group_size = len(group_name) 

base_du = 20 
post_du = 3+6 
epochs = {
    'base':  (0,   int(base_du*fs)),
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du)*fs)) # Combining Post1 and Post2 for visual simplicity
    #'iti': (int((base_du+20+20+3+post_du)*fs), int((base_du+20+20+3+post_du+20)*fs))
}
epoch_durs = {'base':20, 'cs': 20, 'trace': 20, 'us': 3, 'pus': post_du}
epoch_colors = {'base': 'gray', 'cs': '#1f77b4', 'trace': '#2ca02c', 'us': '#d62728', 'pus': 'purple'}   

bins_plot_start = int((20-base_du)*fs) 
bins_plot_end = int(sum(epoch_durs.values())*fs) 

width_mm = 50  # 
# PLot with residuals
test_algori_data = 'post_02_2_resp_cal_new_z_residual'
plot_idx = 1
for i in range(group_size):    
    dpath_test = os.path.join(dpath_cal_all, test_algori_data)
    print(group_name[i])  
    with open(os.path.join(dpath_test, group_name[i] + '_ds_cal_cell_wise_union.pkl'), 'rb') as f:
            ds_data = pickle.load(f)  
    
    # All eligible cells   
    Sig_bin_trial = ds_data['whole'][:, :, bins_plot_start: bins_plot_end]
    Sig_bin_trial = Sig_bin_trial.transpose(1, 0, 2) # From (trials, unit_id, bins) To shape (unit_id, n_trials, n_bins)    

    plot_pca_3d_vector(Sig_bin_trial, [1,2,6], epochs, epoch_colors, fs, width_mm, dpath_plot, f'03_0_{plot_idx}_PCA trajectories_with_T6_vector_{group_name[i]}')
    plot_idx +=1
     
print('All finished************')         

In [ ]:

def plot_pca_3d_vector(data_3d, trials_to_plot, epochs, epoch_colors, fs, width_mm, output_path, title, flag_bs_subtraction=0):
    """
    Plots 3D PCA trajectories for multiple learning stages (e.g., T1, T2, T6) in a single cubic panel.
    Applies visual hierarchy (alpha/linewidth/style) to prevent overlapping clutter.
    Plots the full Naive base trajectory (T1) and fits the 1D PCA vector for the T6 CS/US/pUS trajectory.
    """
    try: set_pub_style()
    except NameError: pass
    
    n_cells, n_trials, n_bins = data_3d.shape   
    
    # 1. TRIAL-BY-TRIAL BASELINE SUBTRACTION
    if flag_bs_subtraction == 1:
        base_key = 'Base' if 'Base' in epochs else 'base'
        base_start, base_end = epochs[base_key]
        for t in range(n_trials):
            trial_base_mean = np.mean(data_3d[:, t, base_start:base_end], axis=1, keepdims=True)
            data_3d[:, t, :] -= trial_base_mean
            
    # 2. TEMPORAL SMOOTHING
    data_smoothed = gaussian_filter1d(data_3d, sigma=2, axis=2)  
    
    # 3. RUN PCA ON CONCATENATED DATA
    data_concat = data_smoothed.transpose(1, 2, 0).reshape(n_trials * n_bins, n_cells)
    pca = PCA(n_components=3)
    pcs_concat = pca.fit_transform(data_concat)
    pcs = pcs_concat.reshape(n_trials, n_bins, 3)   
    var_exp = pca.explained_variance_ratio_ * 100
    
    # 4. SET UP 3D PLOT 
    fig = plt.figure(figsize=(width_mm / 25.4, width_mm / 25.4), dpi=300, layout='constrained')
    ax = fig.add_subplot(111, projection='3d')
    
    style_hierarchy = [
        {'ls': '-',  'lw': 0.5, 'alpha': 1.0,  'marker': '*', 's': 20, 'label': 'Initial'},
        {'ls': '--', 'lw': 0.5, 'alpha': 1.0,  'marker': 'o', 's': 10, 'label': 'Early'},
        {'ls': '-',  'lw': 1.0, 'alpha': 0.25, 'marker': 'o', 's': 20, 'label': 'Late'}
    ]
    
    # 5. PLOT TRAJECTORIES
    starter_bin = int(17 * fs) # Only plot recent 3s base for the standard loop
    for i, t_num in enumerate(trials_to_plot):
        t_idx = t_num - 1 # 0-indexed
        t_pcs = pcs[t_idx]
        sty = style_hierarchy[min(i, len(style_hierarchy)-1)]
        
        # Plot Segments
        for name, (start, end) in epochs.items():
            if name.lower() == 'base':
                start = starter_bin 
            plot_end = min(end + 1, n_bins)
            
            ax.plot(t_pcs[start:plot_end, 0], t_pcs[start:plot_end, 1], t_pcs[start:plot_end, 2], 
                    color=epoch_colors.get(name, 'gray'), linestyle=sty['ls'], linewidth=sty['lw'], alpha=sty['alpha'])
            
        # Plot Start Marker
        ax.scatter(t_pcs[starter_bin, 0], t_pcs[starter_bin, 1], t_pcs[starter_bin, 2], 
                   color='yellow', edgecolor='black', marker=sty['marker'], 
                   s=sty['s'], linewidth=0.4, alpha=sty['alpha'], zorder=10)

    # ---------------------------------------------------------
    # NEW: PLOT FULL NAIVE BASE TRAJECTORY (T1, 0-20s)
    # ---------------------------------------------------------
    t1_idx = trials_to_plot[0] - 1
    base_key = 'base' if 'base' in epochs else ('Base' if 'Base' in epochs else None)
    
    if base_key:
        b_start, b_end = epochs[base_key]
        b_end_plot = min(b_end + 1, n_bins)
        t1_base_traj = pcs[t1_idx, b_start:b_end_plot, :]
        
        ax.plot(t1_base_traj[:, 0], t1_base_traj[:, 1], t1_base_traj[:, 2], 
                color=epoch_colors.get(base_key, 'gray'), linestyle='-', 
                linewidth=1.0, alpha=0.8, zorder=5, label='Naive Base (T1)')

    # ---------------------------------------------------------
    # FIT AND PLOT T6 VECTOR
    # ---------------------------------------------------------
    t6_idx = trials_to_plot[-1] - 1 
    cs_key = 'cs'
    
    if cs_key:
        cs_start = epochs[cs_key][0]
        # Extract T6 data from CS onset to the end of the trial
        t6_eval_traj = pcs[t6_idx, cs_start:, :]
        
        # Fit 1D PCA for the vector fitting
        centroid_t6 = np.mean(t6_eval_traj, axis=0)
        pca_1d = PCA(n_components=1)
        pca_1d.fit(t6_eval_traj - centroid_t6)
        v_t6 = pca_1d.components_[0]
        
        # Project T6 onto the vector to find the min and max coordinates (the length of the line)
        projections = np.dot(t6_eval_traj - centroid_t6, v_t6)
        p_min, p_max = np.min(projections), np.max(projections)
        
        # Add a 10% buffer to the length so the line extends slightly past the trajectory
        buffer = (p_max - p_min) * 0.1
        p_min -= buffer
        p_max += buffer
        
        # Calculate 3D endpoints of the vector
        pt_start = centroid_t6 + p_min * v_t6
        pt_end = centroid_t6 + p_max * v_t6
        
        # Plot the thick dashed vector line
        ax.plot([pt_start[0], pt_end[0]], [pt_start[1], pt_end[1]], [pt_start[2], pt_end[2]], 
                color='black', linestyle='-.', linewidth=1.0, alpha=0.8, zorder=15, label='Learned Vector')
        
        # Optionally mark the centroid of T6
        ax.scatter(*centroid_t6, color='black', marker='X', s=4, zorder=16)

    # 6. AXES FORMATTING (Low Ink Standard)
    try:
        ax.set_box_aspect(None, zoom=0.95)
    except TypeError:
        ax.dist = 12
        
    ax.tick_params(axis='x', labelsize=4, pad=-5)
    ax.tick_params(axis='y', labelsize=4, pad=-5)
    ax.tick_params(axis='z', labelsize=4, pad=-4)

    ax.set_xlabel(f'PC1 ({var_exp[0]:.1f}%)', labelpad=-12) 
    ax.set_ylabel(f'PC2 ({var_exp[1]:.1f}%)', labelpad=-11) 
    ax.set_zlabel(f'PC3 ({var_exp[2]:.1f}%)', labelpad=-26)   

    # BRUTE FORCE 3D BACKGROUND AND GRID FIX
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.set_major_locator(ticker.MaxNLocator(6))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(6)) 
    ax.zaxis.set_major_locator(ticker.MaxNLocator(6)) 

    ax.xaxis.pane.set_edgecolor('none')
    ax.yaxis.pane.set_edgecolor('none')
    ax.zaxis.pane.set_edgecolor('none')

    micro_grid_color = (0.7, 0.7, 0.7, 0.6) 
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis._axinfo["grid"]['color'] = micro_grid_color
        axis._axinfo["grid"]['linewidth'] = 0.2
        axis._axinfo["grid"]['linestyle'] = '-'

    # 8. EXPORT
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))       
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    
    plt.savefig(f"{base_path}.pdf", transparent=True)
    plt.savefig(f"{base_path}.png", dpi=600)
    plt.close()

## 3.1 Trajectory distance among trials (T1 to T6, T2 to T6)--normalized to the distance between "Naive" base and T6 vector

In [ ]:
test_algori_data = 'post_02_2_resp_cal_new_z_residual'
base_du = 20      # Total pre-CS time
stat_base_du = 3  
post_du1 = 3
post_du2 = 6
epochs = {
    'context': (0, int((base_du - stat_base_du) * fs)), 
    # 17s to 20s: The strict 3s origin for statistics
    'base':    (int((base_du - stat_base_du) * fs), int(base_du * fs)), 
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus_1':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du1)*fs)), 
    'pus_2':  (int((base_du+20+20+3+post_du1)*fs), int((base_du+20+20+3+post_du1+post_du2)*fs))
}

epoch_durs = {'base':20, 'cs': 20, 'trace': 20, 'us': 3, 'pus_1': post_du1, 'pus_2': post_du2}

bins_start = int((20-base_du)*fs) # pre-shock 3s
bins_end = int(sum(epoch_durs.values())*fs) # 


height_mm = 35 #  
width_mm = 45
# For EC3
group_name = ['05.EC3-C', '06.EC3-I', '02.CA1-C', '03.CA1-I'] 
group_keys = ['EC3-C', 'EC3-I', 'CA1-C', 'CA1-I']
group_size = len(group_name) # 3

data_3d_group = {}
start_epoch = 'cs'
for i in range(group_size):    
    dpath_test = os.path.join(dpath_cal_all, test_algori_data)
    print(group_name[i])  
    with open(os.path.join(dpath_test, f'{group_name[i]}_ds_cal_animal_wise.pkl'), 'rb') as f:
            ds_data = pickle.load(f)      
    # All eligible cells   
    Sig_bin_trial = []
    for animal_i in range(len(ds_data['whole'])):
        Sig_bin_trial.append(ds_data['whole'][animal_i][:, :, bins_start: bins_end])
    print(len(Sig_bin_trial))
    data_3d_group[group_keys[i]] = Sig_bin_trial
stats_dict = extract_projection_distance_jackknife(data_3d_group, start_epoch, epochs, min_cells=50, fs=5.0)
plot_trajectory_distance(stats_dict, group_keys, 'Orthogonal', width_mm, height_mm, colors_beh_i, dpath_plot, f"03_1_PCA_trajectory_State distance to T6 (normalized to Base-T6)")

print('All finished************')         

In [ ]:
def add_significance_bar(ax, x1, x2, y_max, text, axis_range):
    """
    Draws a flat significance line and returns the 'roof' so brackets stack flawlessly.
    """
    if text == 'ns' or not text:
        return y_max        
    
    line_gap = axis_range * 0.06      
    text_offset = axis_range * 0.02    
    star_height = axis_range * 0.04

    y_line = y_max + line_gap
    # Draw flat horizontal line 
    ax.plot([x1, x2], [y_line, y_line], lw=0.5, c='k')   
    ax.text((x1+x2)*0.5, y_line + text_offset, text, ha='center', va='center', color='k', fontsize=6)
    
    return y_line + text_offset + star_height

def calculate_trajectory_projections(traj_eval, traj_T6, traj_base):
    """
    Fits a 1D vector to the T6 trajectory.
    Normalizes distances using the baseline state before 1st trial(traj_base).
    """
    # 1. Fit the Learned Axis (1st PC of the T6 trajectory)
    centroid_T6 = np.mean(traj_T6, axis=0)
    centered_T6 = traj_T6 - centroid_T6
    
    pca_1d = PCA(n_components=1)
    pca_1d.fit(centered_T6)
    v_T6 = pca_1d.components_[0] 
    
    # 2. Calculate the Baseline Normalization Factors
    centered_base = traj_base - centroid_T6
    proj_base = np.dot(centered_base, v_T6)
    
    # Orthogonal components of the base relative to the T6 vector
    ortho_vec_base = centered_base - np.outer(proj_base, v_T6)
    
    base_ortho_shift = np.mean(np.linalg.norm(ortho_vec_base, axis=1))
    
    # Safety threshold
    if base_ortho_shift < 1e-6: base_ortho_shift = 1.0

    # 3. Project the Evaluation Trajectory (T1 or T2) and T6 itself
    centered_eval = traj_eval - centroid_T6
    proj_eval = np.dot(centered_eval, v_T6)
    
    # Orthogonal Distance
    total_dist_sq = np.sum(centered_eval**2, axis=1)
    ortho_dist = np.sqrt(np.maximum(total_dist_sq - proj_eval**2, 0))

    # 4. Apply Baseline Normalization
    mean_ortho = np.mean(ortho_dist) / base_ortho_shift

    return mean_ortho

def extract_projection_distance_jackknife(data_3d_group, start_epoch, epochs, min_cells=50, fs=5.0):
    """
    Fits a PCA per group using pooled cells.
    Calculates distance relative to the T6 vector, normalized by the Base-to-T6 vector shift.
    """
    stats_dict = {}
    
    eval_start = epochs[start_epoch][0]
    eval_end = epochs['pus_2'][1] 
    
    base_start = epochs['context'][0]
    base_end = epochs['base'][1]
    
    for group_name, animal_list in data_3d_group.items():
        valid_animals = [a for a in animal_list if a.size > 0 and a.shape[1] >= min_cells]
        if not valid_animals:
            continue
            
        animal_cell_counts = [a.shape[1] for a in valid_animals]
        n_trials, _, n_bins = valid_animals[0].shape
        eval_end_safe = min(eval_end, n_bins)
        base_end_safe = min(base_end, n_bins)
        
        master_data = np.concatenate(valid_animals, axis=1) 
        master_smoothed = gaussian_filter1d(master_data, sigma=2, axis=2)
        master_concat = np.transpose(master_smoothed, (1, 0, 2)).reshape(master_data.shape[1], -1)
        
        pca = PCA(n_components=3)
        pca.fit(master_concat.T)
        master_weights = pca.components_ 
        
        group_stats = {
            'ortho_T1': np.zeros(len(valid_animals)), 
            'ortho_T2': np.zeros(len(valid_animals))
        }
        
        current_idx = 0
        for a_idx, n_c in enumerate(animal_cell_counts):
            pseudo_idx = list(range(0, current_idx)) + list(range(current_idx + n_c, master_data.shape[1]))
            
            pseudo_data = master_smoothed[:, pseudo_idx, :]
            pseudo_weights = master_weights[:, pseudo_idx]
            
            pseudo_concat = np.transpose(pseudo_data, (0, 2, 1)).reshape(-1, len(pseudo_idx))
            correction_factor = master_data.shape[1] / len(pseudo_idx)
            
            pseudo_pcs = np.dot(pseudo_concat, (pseudo_weights * correction_factor).T)
            pseudo_pcs = pseudo_pcs.reshape(n_trials, n_bins, 3)
            
            traj_T1 = pseudo_pcs[0, eval_start:eval_end_safe, :]
            traj_T2 = pseudo_pcs[1, eval_start:eval_end_safe, :]
            traj_T6 = pseudo_pcs[-1, eval_start:eval_end_safe, :]
            
            traj_base = pseudo_pcs[0, base_start:base_end_safe, :]
            
            # Calculate projections with Baseline Normalization
            group_stats['ortho_T1'][a_idx] = calculate_trajectory_projections(traj_T1, traj_T6, traj_base)
            group_stats['ortho_T2'][a_idx] = calculate_trajectory_projections(traj_T2, traj_T6, traj_base)
            
            current_idx += n_c
            
        stats_dict[group_name] = group_stats        
    return stats_dict


def plot_trajectory_distance(stats_dict, group_keys, y_label, width_mm, height_mm, colors, output_path, title):
    """
    Combines 4 groups (e.g. EC3-C, EC3-I, CA1-C, CA1-I) into a single 4-column plot.
    With unpaired Mann-Whitney U testing within each region.
    """
    set_pub_style()
    if y_label == 'Orthogonal':
        dict_k1, dict_k2 = 'ortho_T1', 'ortho_T2'
        y_axis_name = 'Trajectory dis. to T6\n(Fraction of Base-T6)'
    else:
        raise ValueError("y_label must be 'Orthogonal'")
        
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    metadata = {
        "Figure_Title": title,
        "Y_Axis_Metric": y_label,
        "Statistics": {}
    }
    
    c_dark = colors[0]
    c_light = lighten_color(c_dark, amount=0.4) if 'lighten_color' in globals() else colors[0]
    i_dark = colors[1]
    i_light = lighten_color(i_dark, amount=0.4) if 'lighten_color' in globals() else colors[1]

    # Map the 4 input keys into regions: (Region_Name, Control_Key, Inactive_Key, X_Shift)
    # X_Shift of 2.5 puts CA1 cleanly next to EC3 (x=1, 2 vs x=3.5, 4.5)
    regions = [
        ('EC3', group_keys[0], group_keys[1], 0.0),
        ('CA1', group_keys[2], group_keys[3], 2.5)
    ]
    
    conditions = [('T1', dict_k1, 1), ('T2', dict_k2, 2)]
    all_vals = []
    
    # 1. EXTRACT DATA & PLOT SCATTERS/MEANS
    for region_name, key_c, key_i, x_shift in regions:
        for cond_name, dict_key, x_base in conditions:
            
            # Retrieve Control and Inactive data for this epoch
            d1 = stats_dict[key_c][dict_key]
            d2 = stats_dict[key_i][dict_key]
            d1 = d1[~np.isnan(d1)]
            d2 = d2[~np.isnan(d2)]
            
            all_vals.extend(d1)
            all_vals.extend(d2)
            
            x_center = x_base + x_shift
            x1 = x_center - 0.15
            x2 = x_center + 0.15
            
            m1, s1 = np.mean(d1), np.std(d1) / np.sqrt(len(d1)) if len(d1)>0 else (np.nan, np.nan)
            m2, s2 = np.mean(d2), np.std(d2) / np.sqrt(len(d2)) if len(d2)>0 else (np.nan, np.nan)
            
            jitter = 0.2
            x1_scatter = x1 + np.random.uniform(-jitter, jitter, size=len(d1))
            x2_scatter = x2 + np.random.uniform(-jitter, jitter, size=len(d2))
            
            # Plot only one label for the legend (assign to the very first EC3 T1 dots)
            l1 = "Control" if (region_name == 'EC3' and cond_name == 'T1') else None
            l2 = "EC5b-Inh" if (region_name == 'EC3' and cond_name == 'T1') else None
            
            ax.scatter(x1_scatter, d1, s=4.0, color=c_light, alpha=0.8, edgecolors='none', zorder=1, label=l1)
            ax.scatter(x2_scatter, d2, s=4.0, color=i_light, alpha=0.8, edgecolors='none', zorder=1, label=l2)
            
            if not np.isnan(m1):
                ax.errorbar(x1, m1, yerr=s1, fmt='o', color=c_dark, elinewidth=0.75, capsize=0, 
                            markersize=4, markeredgecolor='white', markeredgewidth=0.5, zorder=3)
            if not np.isnan(m2):
                ax.errorbar(x2, m2, yerr=s2, fmt='o', color=i_dark, elinewidth=0.75, capsize=0, 
                            markersize=4, markeredgecolor='white', markeredgewidth=0.5, zorder=3)
                            
            # Statistical Testing & Metadata Export
            p_val = np.nan
            if len(d1) > 2 and len(d2) > 2:
                _, p_val = stats.mannwhitneyu(d1, d2, alternative='two-sided')
                
            metadata["Statistics"][f"{region_name}_{cond_name}"] = {
                "N_Control": int(len(d1)),
                "N_Inactive": int(len(d2)),
                "Mean_Control": float(m1) if not np.isnan(m1) else None,
                "Mean_Inactive": float(m2) if not np.isnan(m2) else None,
                "p_value_mannwhitney": float(p_val) if not np.isnan(p_val) else None
            }

    # 2. ADD SIGNIFICANCE BRACKETS 
    data_range = max(all_vals) - min(all_vals) if all_vals else 1.0
    
    for region_name, key_c, key_i, x_shift in regions:
        for cond_name, dict_key, x_base in conditions:
            p_val = metadata["Statistics"][f"{region_name}_{cond_name}"]["p_value_mannwhitney"]
            star = get_asterisks(p_val)
            
            if star != 'ns':
                x_center = x_base + x_shift
                x1, x2 = x_center - 0.15, x_center + 0.15
                
                # Fetch local max to sit the bracket right above the highest dot for this comparison
                d1 = stats_dict[key_c][dict_key]
                d2 = stats_dict[key_i][dict_key]
                local_max = max(np.max(d1[~np.isnan(d1)]), np.max(d2[~np.isnan(d2)]))
                
                add_significance_bar(ax, x1, x2, local_max, star, data_range)

    # 3. AXES FORMATTING
    ax.set_xticks([1, 2, 3.5, 4.5])
    # Label nicely integrates region and trial
    ax.set_xticklabels(['EC3\nT1', 'EC3\nT2', 'CA1\nT1', 'CA1\nT2'], fontsize=7)
    ax.set_ylabel(y_axis_name, labelpad=1)
    
    # Separation line between EC3 and CA1
    ax.axvline(2.75, color='gray', linestyle=':', linewidth=0.5, alpha=0.5, zorder=0)
    
    ax.set_xlim(0.4, 5.1)
    
    if all_vals:
        global_max = max(all_vals)
        global_min = min(all_vals)
        y_range = global_max - global_min
        if y_range == 0: y_range = 1.0
        ax.set_ylim(max(0, global_min - y_range * 0.1), global_max + y_range * 0.3)
        
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    ax.tick_params(axis='both', length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    
    # Legend - Place neatly at the top left without blocking data
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    #ax.legend(by_label.values(), by_label.keys(), frameon=False, fontsize=5, loc='upper left', bbox_to_anchor=(0.0, 1.15), ncol=2)
    
    # 4. EXPORT
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))       
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 3.2 Trajectory similarity among trials for US and pUS epoch--centered DTW distance (dynamic time warping)

In [ ]:
test_algori_data = 'post_02_2_resp_cal_new_z_residual'
base_du = 20      # Total pre-CS time
stat_base_du = 3 
post_du1 = 3
post_du2 = 6
epochs = {
    'context': (0, int((base_du - stat_base_du) * fs)), 
    # 17s to 20s: The strict 3s origin for statistics
    'base':    (int((base_du - stat_base_du) * fs), int(base_du * fs)), 
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus_1':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du1)*fs)),
    'pus_2':  (int((base_du+20+20+3+post_du1)*fs), int((base_du+20+20+3+post_du1+post_du2)*fs))
}

epoch_durs = {'base':20, 'cs': 20, 'trace': 20, 'us': 3, 'pus_1': post_du1, 'pus_2': post_du2}

bins_start = int((20-base_du)*fs) # pre-shock 3s
bins_end = int(sum(epoch_durs.values())*fs) # 

height_mm = 30
width_mm = 40
# For EC3
group_name = ['05.EC3-C', '06.EC3-I']
group_keys = ['EC3-C', 'EC3-I']
group_size = len(group_name) # 3

data_3d_group = {}
for i in range(group_size):    
    dpath_test = os.path.join(dpath_cal_all, test_algori_data)
    print(group_name[i])  
    with open(os.path.join(dpath_test, f'{group_name[i]}_ds_cal_animal_wise.pkl'), 'rb') as f:
            ds_data = pickle.load(f)      
    # All eligible cells   
    Sig_bin_trial = []
    for animal_i in range(len(ds_data['whole'])):
        Sig_bin_trial.append(ds_data['whole'][animal_i][:, :, bins_start: bins_end])
    print(len(Sig_bin_trial))
    data_3d_group[group_keys[i]] = Sig_bin_trial

stats_dict = extract_trajectory_similarity_jackknife(data_3d_group, epochs)
plot_trial_trajectory_similarity(stats_dict, group_keys, 'Mean-Centered DTW', width_mm, height_mm, colors_beh_i, dpath_plot, 
                                f"03_2_Trajectory_similarity_centered_DTW_distance_{group_keys[0][0:3]}")

# For CA1
group_name = ['02.CA1-C', '03.CA1-I']
group_keys = ['CA1-C', 'CA1-I']
group_size = len(group_name) 

data_3d_group = {}
for i in range(group_size):    
    dpath_test = os.path.join(dpath_cal_all, test_algori_data)
    print(group_name[i])  
    with open(os.path.join(dpath_test, f'{group_name[i]}_ds_cal_animal_wise.pkl'), 'rb') as f:
            ds_data = pickle.load(f)      
    # All eligible cells   
    Sig_bin_trial = []
    for animal_i in range(len(ds_data['whole'])):
        Sig_bin_trial.append(ds_data['whole'][animal_i][:, :, bins_start: bins_end])
    print(len(Sig_bin_trial))
    data_3d_group[group_keys[i]] = Sig_bin_trial
stats_dict = extract_trajectory_similarity_jackknife(data_3d_group, epochs)
plot_trial_trajectory_similarity(stats_dict, group_keys, 'Mean-Centered DTW', width_mm, height_mm, colors_beh_i, dpath_plot, 
                                f"03_2_Trajectory_similarity_centered_DTW_distance_{group_keys[0][0:3]}")

print('All finished************')         

In [ ]:

def add_significance_bar(ax, x1, x2, y_max, text, data_range):
    """
    Draws a flat significance line and returns the 'roof' so brackets stack flawlessly.
    Uses data_range to compute offsets independently of current ax limits.
    """
    if text == 'ns' or not text:
        return y_max        
    
    line_gap = data_range * 0.09       # Gap from data to line
    text_offset = data_range * 0.02    # Gap from line to text
    star_height = data_range * 0.06    # Space for text itself

    y_line = y_max + line_gap
    # Draw flat horizontal line 
    ax.plot([x1, x2], [y_line, y_line], lw=0.75, c='k')   
    ax.text((x1+x2)*0.5, y_line + text_offset, text, ha='center', va='center', color='k', fontsize=7)
    
    return y_line + text_offset + star_height

def extract_trajectory_similarity_jackknife(data_3d_group, epochs, min_cells=50, fs=5.0):
    """
    Fits a Master PCA for each group independently.
    Calculates Mean-Centered DTW.
    """
    stats_dict = {}
    eval_start = epochs['us'][0]
    
    for group_name, animal_list in data_3d_group.items():
        valid_animals = [a for a in animal_list if a.size > 0 and a.shape[1] >= min_cells]
        if not valid_animals:
            continue
            
        n_animals = len(valid_animals)
        animal_cell_counts = [a.shape[1] for a in valid_animals]
        n_trials, _, n_bins = valid_animals[0].shape
        eval_end = min(epochs['pus_2'][1], n_bins)
        
        # Pool cells for the Group Master PCA
        master_data = np.concatenate(valid_animals, axis=1) 
        master_smoothed = gaussian_filter1d(master_data, sigma=2, axis=2)
        master_concat = np.transpose(master_smoothed, (1, 0, 2)).reshape(master_data.shape[1], -1)
        
        pca = PCA(n_components=3)
        pca.fit(master_concat.T)
        master_weights = pca.components_ 
        
        dtw_T1_T6 = np.zeros(n_animals)
        dtw_T2_T6 = np.zeros(n_animals)
  
        
        current_idx = 0
        for a_idx, n_c in enumerate(animal_cell_counts):
            pseudo_idx = list(range(0, current_idx)) + list(range(current_idx + n_c, master_data.shape[1]))
            
            pseudo_data = master_smoothed[:, pseudo_idx, :]
            pseudo_weights = master_weights[:, pseudo_idx]
            
            pseudo_concat = np.transpose(pseudo_data, (0, 2, 1)).reshape(-1, len(pseudo_idx))
            correction_factor = master_data.shape[1] / len(pseudo_idx)
            
            pseudo_pcs = np.dot(pseudo_concat, (pseudo_weights * correction_factor).T)
            pseudo_pcs = pseudo_pcs.reshape(n_trials, n_bins, 3)
            
            traj_T1 = pseudo_pcs[0, eval_start:eval_end, :]
            traj_T2 = pseudo_pcs[1, eval_start:eval_end, :]
            traj_T6 = pseudo_pcs[-1, eval_start:eval_end, :] 

            # MEAN-CENTERING
            traj_T1_centered = traj_T1 - np.mean(traj_T1, axis=0)
            traj_T2_centered = traj_T2 - np.mean(traj_T2, axis=0)
            traj_T6_centered = traj_T6 - np.mean(traj_T6, axis=0)

            # DTW
            dist_1_6, _ = fastdtw(traj_T1_centered, traj_T6_centered, dist=euclidean)
            dist_2_6, _ = fastdtw(traj_T2_centered, traj_T6_centered, dist=euclidean)
            
            bins_evaluated = len(traj_T6_centered)
            dtw_T1_T6[a_idx] = dist_1_6 / bins_evaluated
            dtw_T2_T6[a_idx] = dist_2_6 / bins_evaluated           

            current_idx += n_c
            
        stats_dict[group_name] = {
            'dtw_T1_T6': dtw_T1_T6,
            'dtw_T2_T6': dtw_T2_T6}
        
    return stats_dict


def plot_trial_trajectory_similarity(stats_dict, group_keys, y_label, width_mm, height_mm, colors, output_path, title):
    """
    Plots paired within-group comparisons and unpaired between-group comparisons for 2 groups.
    Automatically stacks significance brackets and exports statistics to JSON.
    """
    set_pub_style()
    
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    if y_label == 'Mean-Centered DTW':
        k_1, k_2 = 'dtw_T1_T6', 'dtw_T2_T6'
        y_axis_name = 'Trajectory diff. to T6\n(DTW)'
    else:
        raise ValueError("Invalid y_label")
    
    # Metadata for JSON
    metadata = {"Figure_Title": title, "Y_Axis_Metric": y_label, "Statistics": {}}
    
    # Group 1: x=1, x=2. Group 2: x=3.5, x=4.5
    x_pos = {
        group_keys[0]: [1, 2],
        group_keys[1]: [3.5, 4.5]
    }
    
    c_dark, c_light = colors[0], (lighten_color(colors[0], 0.4) if 'lighten_color' in globals() else colors[0])
    i_dark, i_light = colors[1], (lighten_color(colors[1], 0.4) if 'lighten_color' in globals() else colors[1])
    group_colors = {group_keys[0]: (c_dark, c_light), group_keys[1]: (i_dark, i_light)}
    
    all_vals = []
    
    # 1. Plot Data points, Lines, and Errors
    for g_idx, g_key in enumerate(group_keys):
        y1 = stats_dict[g_key][k_1]
        y2 = stats_dict[g_key][k_2]
        
        all_vals.extend(y1)
        all_vals.extend(y2)
        
        n_samples = len(y1)
        x1_base, x2_base = x_pos[g_key]
        
        dark_c, light_c = group_colors[g_key]
        
        # Paired lines
        for i in range(n_samples):
            ax.plot([x1_base, x2_base], [y1[i], y2[i]], color='gray', alpha=0.3, linewidth=0.5, zorder=1)
            
        # Scatter Dots
        jitter = 0.05
        x1_scatter = x1_base + np.random.uniform(-jitter, jitter, size=n_samples)
        x2_scatter = x2_base + np.random.uniform(-jitter, jitter, size=n_samples)
        
        ax.scatter(x1_scatter, y1, s=4.0, color=light_c, alpha=0.8, edgecolors='none', zorder=2, label=g_key)
        ax.scatter(x2_scatter, y2, s=4.0, color=light_c, alpha=0.8, edgecolors='none', zorder=2)
        
        # Means and Errors
        m1, s1 = np.mean(y1), np.std(y1) / np.sqrt(n_samples)
        m2, s2 = np.mean(y2), np.std(y2) / np.sqrt(n_samples)
        
        ax.errorbar(x1_base, m1, yerr=s1, fmt='o', color=dark_c, elinewidth=0.75, capsize=0, 
                    markersize=4, markeredgecolor='white', markeredgewidth=0.5, zorder=3)
        ax.errorbar(x2_base, m2, yerr=s2, fmt='o', color=dark_c, elinewidth=0.75, capsize=0, 
                    markersize=4, markeredgecolor='white', markeredgewidth=0.5, zorder=3)
        ax.plot([x1_base, x2_base], [m1, m2], color='black', linewidth=1.0, zorder=3)
        
        # JSON Paired Stats (Wilcoxon)
        _, p_paired = stats.wilcoxon(y1, y2, alternative='two-sided')
        metadata["Statistics"][f"{g_key}_Paired"] = {
            "N": n_samples, "Mean_T1_T6": float(m1), "Mean_T2_T6": float(m2), "p_value_wilcoxon": float(p_paired)
        }

    # 2. Add Significance Brackets sequentially
    data_range = max(all_vals) - min(all_vals) if all_vals else 1.0
    current_roof = max(all_vals) + (data_range * 0.05)
    
    # Layer 1: Paired Comparisons (Wilcoxon)
    roof_g1 = current_roof
    roof_g2 = current_roof
    
    p_g1 = metadata["Statistics"][f"{group_keys[0]}_Paired"]["p_value_wilcoxon"]
    if get_asterisks(p_g1) != 'ns':
        roof_g1 = add_significance_bar(ax, x_pos[group_keys[0]][0], x_pos[group_keys[0]][1], current_roof, get_asterisks(p_g1), data_range)
        
    p_g2 = metadata["Statistics"][f"{group_keys[1]}_Paired"]["p_value_wilcoxon"]
    if get_asterisks(p_g2) != 'ns':
        roof_g2 = add_significance_bar(ax, x_pos[group_keys[1]][0], x_pos[group_keys[1]][1], current_roof, get_asterisks(p_g2), data_range)
        
    # Layer 2: Unpaired T1-T6 (Mann-Whitney)
    current_roof = max(roof_g1, roof_g2)
    y1_g1, y1_g2 = stats_dict[group_keys[0]][k_1], stats_dict[group_keys[1]][k_1]
    _, p_unpaired_t1 = stats.mannwhitneyu(y1_g1, y1_g2, alternative='two-sided')
    metadata["Statistics"]["Unpaired_T1_T6"] = {"p_value_mannwhitney": float(p_unpaired_t1)}
    
    if get_asterisks(p_unpaired_t1) != 'ns':
        current_roof = add_significance_bar(ax, x_pos[group_keys[0]][0], x_pos[group_keys[1]][0], current_roof, get_asterisks(p_unpaired_t1), data_range)

    # Layer 3: Unpaired T2-T6 (Mann-Whitney)
    y2_g1, y2_g2 = stats_dict[group_keys[0]][k_2], stats_dict[group_keys[1]][k_2]
    _, p_unpaired_t2 = stats.mannwhitneyu(y2_g1, y2_g2, alternative='two-sided')
    metadata["Statistics"]["Unpaired_T2_T6"] = {"p_value_mannwhitney": float(p_unpaired_t2)}
    
    if get_asterisks(p_unpaired_t2) != 'ns':
        current_roof = add_significance_bar(ax, x_pos[group_keys[0]][1], x_pos[group_keys[1]][1], current_roof, get_asterisks(p_unpaired_t2), data_range)

    # 3. Axes Formatting
    ax.set_xticks([1, 2, 3.5, 4.5])
    ax.set_xticklabels(['T1', 'T2', 'T1', 'T2'], fontsize=7)
    ax.set_ylabel(y_axis_name, labelpad=1)
    
    # Pad x-axis to prevent crowding against spines
    ax.set_xlim(0.3, 5.2)
    
    # Dynamically scale Y-axis to fit all brackets
    ax.set_ylim(min(all_vals) - data_range*0.1, current_roof + data_range*0.1)
    
    ax.yaxis.set_major_locator(ticker.MultipleLocator(10)) #  MaxNLocator(nbins=4)
    ax.tick_params(axis='both', length=2, pad=1)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_linewidth(0.5)
    
    # Clean up legend (remove duplicate labels created in loop)
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    #ax.legend(by_label.values(), by_label.keys(), frameon=False, fontsize=5, loc='upper left', bbox_to_anchor=(0, 1.15))

    # 4. Export
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")#transparent=True
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 3.3 PCA abstract plot of CS state centered analysis

In [ ]:
# 3D plotting with residuals
group_name = ['01.EC5b', '02.CA1-C', '03.CA1-I', '05.EC3-C', '06.EC3-I'] 
group_size = len(group_name) # 3

base_du = 20 
post_du = 3+6 
epochs = {
    'base':  (0,   int(base_du*fs)),
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du)*fs)) # Combining Post1 and Post2 for visual simplicity
}
epoch_durs = {'base':20, 'cs': 20, 'trace': 20, 'us': 3, 'pus': post_du}
epoch_colors = {'base': 'gray', 'cs': '#1f77b4', 'trace': '#2ca02c', 'us': '#d62728', 'pus': 'purple'}   

bins_plot_start = int((20-base_du)*fs) # pre-shock 3s
bins_plot_end = int(sum(epoch_durs.values())*fs) # 

width_mm = 45  # 
height_mm = 30
# PLot with residuals
test_algori_data = 'post_02_2_resp_cal_new_z_residual'
plot_idx = 1
for i in range(group_size):    
    dpath_test = os.path.join(dpath_cal_all, test_algori_data)
    print(group_name[i])  
    with open(os.path.join(dpath_test, group_name[i] + '_ds_cal_cell_wise_union.pkl'), 'rb') as f:
            ds_data = pickle.load(f)  
    
    # All eligible cells   
    Sig_bin_trial = ds_data['whole'][:, :, bins_plot_start: bins_plot_end]
    Sig_bin_trial = Sig_bin_trial.transpose(1, 0, 2) # From (trials, unit_id, bins) To shape (unit_id, n_trials, n_bins)    

    if group_name[i] == '01.EC5b':
        flag_size =1
    else:
        flag_size = 0
    plot_pca_3d_cs_abstract(Sig_bin_trial, [1,2,6], epochs, epoch_colors, fs, width_mm, height_mm, dpath_plot, f'03_3_{plot_idx}_PCA abstract with CS state_all_eligible_cells_residuals-{group_name[i]}', flag_size)
    plot_idx +=1
print('All finished************')         

In [ ]:
def plot_pca_3d_cs_abstract(data_3d, trials_to_plot, epochs, epoch_colors, fs, width_mm, height_mm, output_path, title, flag_size=0):
    """
    Plots an abstracted, floating 1x3 grid of 3D PCA trajectories.
    Isolates Trial 1, Trial 2, and Trial 6 in separate panels with no axes or grids,
    focusing purely on the geometric relationship between the CS Centroid and the US/pUS paths.
    """
    set_pub_style()
    n_cells, n_trials, n_bins = data_3d.shape   
    
    # 2. TEMPORAL SMOOTHING & PCA
    data_smoothed = gaussian_filter1d(data_3d, sigma=2, axis=2)  
    data_concat = data_smoothed.transpose(1, 2, 0).reshape(n_trials * n_bins, n_cells)
    
    pca = PCA(n_components=3)
    pcs_concat = pca.fit_transform(data_concat)
    pcs = pcs_concat.reshape(n_trials, n_bins, 3)   
    
    # 3. CALCULATE GLOBAL LIMITS FOR SCALING
    # Even though axes are invisible, lock the spatial scale across the 3 plots
    # so the visual size of the trajectories represents real mathematical distance.
    plot_points = []
    for t_num in trials_to_plot:
        t_pcs = pcs[t_num - 1]
        
        cs_s, cs_e = epochs['cs']
        cs_centroid = np.mean(t_pcs[cs_s:cs_e, :], axis=0)
        
        us_s, pus_e = epochs['us'][0], epochs['pus'][1]
        us_pus_traj = t_pcs[us_s:pus_e, :]
        
        plot_points.append(cs_centroid.reshape(1, 3))
        plot_points.append(us_pus_traj)
        
    all_points = np.vstack(plot_points)
    x_min, x_max = all_points[:, 0].min(), all_points[:, 0].max()
    y_min, y_max = all_points[:, 1].min(), all_points[:, 1].max()
    z_min, z_max = all_points[:, 2].min(), all_points[:, 2].max()
    
    # Add a tighter 5% padding (instead of 10%)
    if flag_size==0:
        pad_x, pad_y, pad_z = (x_max-x_min)*0.12, (y_max-y_min)*0.12, (z_max-z_min)*0.12
    else:
        pad_x, pad_y, pad_z = (x_max-x_min)*0.2, (y_max-y_min)*0.2, (z_max-z_min)*0.2
    # 4. SET UP 1x3 ABSTRACT PLOT (e.g., 80mm width, 30mm height)
    fig = plt.figure(figsize=(width_mm / 25.4, height_mm / 25.4), dpi=300, layout='constrained')
    
    # 5. PLOT EACH TRIAL IN ITS OWN PANEL
    for col, t_num in enumerate(trials_to_plot):
        ax = fig.add_subplot(1, 3, col + 1, projection='3d')
        t_idx = t_num - 1 
        t_pcs = pcs[t_idx]
        
        # A. Calculate and Plot CS Centroid (Clear, beautiful circle)
        cs_s, cs_e = epochs['cs']
        cs_centroid = np.mean(t_pcs[cs_s:cs_e, :], axis=0)
        
        ax.scatter(*cs_centroid, marker='o', s=30, 
                   facecolor=epoch_colors['cs'], edgecolor='white', 
                   linewidth=0.8, alpha=1.0, zorder=10, depthshade=False)
                   
        # B. Plot the "Tether" (A faint line showing exactly how the US jumped from the CS)
        us_s = epochs['us'][0]
        us_onset_point = t_pcs[us_s, :]
        ax.plot([cs_centroid[0], us_onset_point[0]], 
                [cs_centroid[1], us_onset_point[1]], 
                [cs_centroid[2], us_onset_point[2]], 
                color='gray', linestyle=':', linewidth=0.8, alpha=0.6, zorder=1)
        
        # C. Plot US and pUS Trajectories (Solid, bold lines for the abstract look)
        for name in ['us', 'pus']:
            if name not in epochs: continue
            start, end = epochs[name]
            plot_end = min(end + 1, n_bins)
            
            ax.plot(t_pcs[start:plot_end, 0], t_pcs[start:plot_end, 1], t_pcs[start:plot_end, 2], 
                    color=epoch_colors[name], linestyle='-', 
                    linewidth=0.4, alpha=0.9, zorder=5)

        # D. ENFORCE INVISIBLE, LOCKED AXES
        ax.set_xlim(x_min - pad_x, x_max + pad_x)
        ax.set_ylim(y_min - pad_y, y_max + pad_y)
        ax.set_zlim(z_min - pad_z, z_max + pad_z)
        
        # Force the camera to zoom in and eliminate Matplotlib's default 3D spherical margins
        try:
            ax.set_box_aspect((1, 1, 1), zoom=1.5)
        except TypeError:
            ax.dist = 6  # Fallback for older versions of matplotlib
            
        # Completely turn off the bounding box, panes, and ticks
        ax.set_axis_off()
        
        # Simple text label for the trial
        #ax.set_title(f"Trial {t_num}",  fontweight='bold', pad=0)

    # 6. MINIMAL GLOBAL LEGEND (Placed perfectly in the bottom center)
    custom_lines = [
        Line2D([0], [0], color='none', marker='o', markerfacecolor=epoch_colors['cs'], markeredgecolor='white', markersize=6, label='CS Centroid'),
        Line2D([0], [0], color=epoch_colors['us'], lw=2, label='US Trajectory'),
        Line2D([0], [0], color=epoch_colors['pus'], lw=2, label='pUS Trajectory')
    ]
    
    #fig.legend(handles=custom_lines, loc='lower center', bbox_to_anchor=(0.5, -0.15),
    #           ncol=3, frameon=False)

    # 7. EXPORT
    base_path = os.path.join(output_path, title.replace(' ', '_'))       
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)   
    
    plt.savefig(f"{base_path}.pdf", transparent=True , bbox_inches=None)
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()

## 4.1 Group comparison of phase_portrait to the association axis---for CA1

In [ ]:
# 3D plotting with residuals for CA1
group_name = ['02.CA1-C', '03.CA1-I']
group_size = len(group_name) # 3
group_key = ['CA1-C', 'CA1-I']

base_du = 20 
post_du = 3+6
epochs = {
    'base':  (0,   int(base_du*fs)),
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du)*fs)) # Combining Post1 and Post2 for visual simplicity
}
epoch_durs = {'base':20, 'cs': 20, 'trace': 20, 'us': 3, 'pus': post_du}
epoch_colors = {'base': 'gray', 'cs': '#1f77b4', 'trace': '#2ca02c', 'us': '#d62728', 'pus': 'purple'}   

bins_plot_start = int((20-base_du)*fs) 
bins_plot_end = int(sum(epoch_durs.values())*fs) # 

# PLot with residuals
test_algori_data = 'post_02_2_resp_cal_new_z_residual'

data_3d_group = {}
for i in range(group_size):    
    dpath_test = os.path.join(dpath_cal_all, test_algori_data)
    print(group_name[i])  
    with open(os.path.join(dpath_test, f'{group_name[i]}_ds_cal_animal_wise.pkl'), 'rb') as f:
            ds_data = pickle.load(f)  
    
    # All eligible cells   
    Sig_bin_trial = []
    for animal_i in range(len(ds_data['whole'])):
        #if ds_data['whole'][animal_i].shape[1] > 300:
        Sig_bin_trial.append(ds_data['whole'][animal_i][:, :, bins_plot_start: bins_plot_end])
    print(len(Sig_bin_trial))
    data_3d_group[group_name[i]] = Sig_bin_trial
        
width_mm = 80  # 
height_mm = 30
colors = ['#4091cf', '#e1703c', '#8cba54']
trials_to_plot = [1,2,6] 
plot_pca_phase_portrait_groups_jackknife(data_3d_group, trials_to_plot, epochs, group_key, fs, width_mm, height_mm, colors, dpath_plot, 
                     f'04_1_phase_portrait_trajectory_all trials_all_eligible_cells_residuals_jack')
                            
print('All finished************')         

In [ ]:
def plot_pca_phase_portrait_groups_jackknife(data_3d_group, trials_to_plot, epochs, group_key, fs, width_mm, height_mm, colors, output_path, title):
    """
    Jackknifed PCA Phase Portraits with Root-Cause Denoising.
    Method:
      1. Pools group cells for a Master PCA, uses Jackknife resampling for projection.
      2. Rejects trials where vec_length < 1.0 (insufficient state separation).
      3. Uses mean plotting
    """
    try: set_pub_style()
    except NameError: pass

    # 1. DEFINE EPOCH BOUNDARIES FOR EVALUATION
    cs_s, cs_e = epochs['cs']
    us_s = epochs['us'][0]
    us_e = epochs['us'][1]
    pus_e = epochs['pus'][1]
    
    us_len = us_e - us_s
    
    # 2. EXTRACT DATA WITH ROOT-CAUSE ARTIFACT REJECTION
    group_stats_raw = {}
    artifact_log = {grp: 0 for grp in data_3d_group.keys()}
    
    for grp_name, animals_ls in data_3d_group.items():
        group_stats_raw[grp_name] = {t_num: {"X": [], "Y": []} for t_num in trials_to_plot}
        
        # --- NEW: MASTER PCA FOR JACKKNIFE ---
        valid_animals = [a for a in animals_ls if a is not None and a.size > 0 and a.shape[1] >= 3]
        if not valid_animals:
            continue
            
        animal_cell_counts = [a.shape[1] for a in valid_animals]
        n_trials, _, n_bins = valid_animals[0].shape
        
        # Pool all valid cells for the Master PCA
        master_data = np.concatenate(valid_animals, axis=1) 
        master_smoothed = gaussian_filter1d(master_data, sigma=2, axis=2)
        master_concat = np.transpose(master_smoothed, (1, 0, 2)).reshape(master_data.shape[1], -1)
        
        pca = PCA(n_components=3)
        pca.fit(master_concat.T)
        master_weights = pca.components_ 
        
        current_idx = 0
        for a_idx, n_c in enumerate(animal_cell_counts):
            # Jackknife: Identify indices excluding the current animal
            pseudo_idx = list(range(0, current_idx)) + list(range(current_idx + n_c, master_data.shape[1]))
            
            pseudo_data = master_smoothed[:, pseudo_idx, :]
            pseudo_weights = master_weights[:, pseudo_idx]
            
            pseudo_concat = np.transpose(pseudo_data, (0, 2, 1)).reshape(-1, len(pseudo_idx))
            correction_factor = master_data.shape[1] / len(pseudo_idx)
            
            # Project Pseudo-Population into Master PCA space
            pseudo_pcs = np.dot(pseudo_concat, (pseudo_weights * correction_factor).T)
            pca_trials = pseudo_pcs.reshape(n_trials, n_bins, 3)
            
            current_idx += n_c
            
            # --- DOWNSTREAM PROCESSING (UNMODIFIED) ---
            for t_num in trials_to_plot:
                t_idx = t_num - 1
                if t_idx >= n_trials: continue
                
                t_pcs = pca_trials[t_idx]
                
                cs_centroid = np.mean(t_pcs[cs_s:cs_e, :], axis=0)
                traj_eval_us = t_pcs[us_s:us_e, :]
                dists_to_cs_us = np.linalg.norm(traj_eval_us - cs_centroid, axis=1)
                max_dist_idx = np.argmax(dists_to_cs_us)
                
                traj_1 = traj_eval_us[:max_dist_idx + 1, :]
                us_centroid = np.mean(traj_1, axis=0)
                
                vector_cs_us = us_centroid - cs_centroid
                vec_length = np.linalg.norm(vector_cs_us)
                
                # --- ROOT-CAUSE DENOISING TIER 1 ---
                if vec_length < 1.0:
                    artifact_log[grp_name] += 1
                    continue
                
                traj_eval_all = t_pcs[us_s:pus_e, :]
                V = traj_eval_all - cs_centroid
                X = np.dot(V, vector_cs_us) / (vec_length ** 2)
    
                V_parallel = np.outer(X, vector_cs_us)
                V_ortho = V - V_parallel
                Y = np.linalg.norm(V_ortho, axis=1) / vec_length # with normalization
                
                # Median filter to erase single-bin transient spikes
                X = median_filter(X, size=3)
                Y = median_filter(Y, size=3)
                
                group_stats_raw[grp_name][t_num]["X"].append(X)
                group_stats_raw[grp_name][t_num]["Y"].append(Y)

    # 3. METRICS EXTRACTION
    group_stats = group_stats_raw 
    metrics_log = {
        "Figure_Title": title,
        "Trials": trials_to_plot,
        "Quantification": {}
    }
    
    plot_limits = {t_num: {"x_min": 0, "x_max": 1, "y_max": 1} for t_num in trials_to_plot}
    
    for t_num in trials_to_plot:
        pooled_x, pooled_y = [], []
        for grp_name in group_stats.keys():
            if len(group_stats[grp_name][t_num]["X"]) > 0:
                pooled_x.extend(np.concatenate(group_stats[grp_name][t_num]["X"]))
                pooled_y.extend(np.concatenate(group_stats[grp_name][t_num]["Y"]))
                
        if len(pooled_x) > 0:
            plot_limits[t_num]["x_min"] = np.percentile(pooled_x, 10)
            plot_limits[t_num]["x_max"] = np.percentile(pooled_x, 90)
            plot_limits[t_num]["y_max"] = np.percentile(pooled_y, 90) 
            
            for grp_name in group_stats.keys():
                if grp_name not in metrics_log["Quantification"]:
                    metrics_log["Quantification"][grp_name] = {t: {"US_VarRatio": [], "pUS_VarRatio": [], "US_AUC": [], "pUS_AUC": []} for t in trials_to_plot}
                
                for i in range(len(group_stats[grp_name][t_num]["X"])):
                    clean_X = group_stats[grp_name][t_num]["X"][i]
                    clean_Y = group_stats[grp_name][t_num]["Y"][i]
                    
                    X_us, Y_us = clean_X[:us_len], clean_Y[:us_len]
                    X_pus, Y_pus = clean_X[us_len:], clean_Y[us_len:]
                    
                    vr_us = np.var(Y_us) / (np.var(X_us) + 1e-8)
                    vr_pus = np.var(Y_pus) / (np.var(X_pus) + 1e-8)
                    auc_us = np.sum(Y_us) / fs
                    auc_pus = np.sum(Y_pus) / fs
                    
                    metrics_log["Quantification"][grp_name][t_num]["US_VarRatio"].append(float(vr_us))
                    metrics_log["Quantification"][grp_name][t_num]["pUS_VarRatio"].append(float(vr_pus))
                    metrics_log["Quantification"][grp_name][t_num]["US_AUC"].append(float(auc_us))
                    metrics_log["Quantification"][grp_name][t_num]["pUS_AUC"].append(float(auc_pus))

    # 4. STATISTICAL TESTING (Mann-Whitney U)
    group_names = list(data_3d_group.keys())
    metrics_log["Statistics_MWU"] = {}
    
    if len(group_names) >= 2:
        g1, g2 = group_names[0], group_names[1]
        metrics_log["Statistics_MWU"][f"{g1}_vs_{g2}"] = {}
        
        for t_num in trials_to_plot:
            metrics_log["Statistics_MWU"][f"{g1}_vs_{g2}"][f"Trial_{t_num}"] = {}
            for metric in ["US_VarRatio", "pUS_VarRatio", "US_AUC", "pUS_AUC"]:
                if t_num in metrics_log["Quantification"][g1]:
                    data1 = metrics_log["Quantification"][g1][t_num][metric]
                    data2 = metrics_log["Quantification"][g2][t_num][metric]
                    
                    if len(data1) > 2 and len(data2) > 2:
                        _, p_val = stats.mannwhitneyu(data1, data2, alternative='two-sided')
                        metrics_log["Statistics_MWU"][f"{g1}_vs_{g2}"][f"Trial_{t_num}"][metric] = float(p_val)
                    else:
                        metrics_log["Statistics_MWU"][f"{g1}_vs_{g2}"][f"Trial_{t_num}"][metric] = "Not enough data"

    # 5. SET UP THE PLOT
    n_panels = len(trials_to_plot)
    fig, axes = plt.subplots(1, n_panels, 
                             figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    if n_panels == 1: axes = [axes]
    color_map = {grp: colors[i % len(colors)] for i, grp in enumerate(group_names)}

    # 6. PLOT PHASE PORTRAITS
    for ax_idx, t_num in enumerate(trials_to_plot):
        ax = axes[ax_idx]
        
        pad_x = (plot_limits[t_num]["x_max"] - plot_limits[t_num]["x_min"]) * 0.1
        ax_x_min = plot_limits[t_num]["x_min"] - pad_x
        ax_x_max = plot_limits[t_num]["x_max"] + pad_x
        
        ax.scatter([0, 1], [0, 0], color='black', s=10, zorder=2) 

        for idx, grp_name in enumerate(group_names):
            if grp_name not in group_stats: continue
            list_x = group_stats[grp_name][t_num]["X"]
            list_y = group_stats[grp_name][t_num]["Y"]
            if len(list_x) == 0: continue
            
            for anim_x, anim_y in zip(list_x, list_y):
                smooth_x = gaussian_filter1d(anim_x, sigma=1.0)
                smooth_y = gaussian_filter1d(anim_y, sigma=1.0)
                ax.plot(smooth_x, smooth_y, color=color_map[grp_name], alpha=0.15, lw=0.5, zorder=3)
            
            mean_x = gaussian_filter1d(np.nanmean(np.array(list_x), axis=0), sigma=1.0)
            mean_y = gaussian_filter1d(np.nanmean(np.array(list_y), axis=0), sigma=1.0)
            ax.plot(mean_x, mean_y, color=color_map[grp_name], lw=0.75, label=group_key[idx], zorder=4)
            
            if us_len < len(mean_x):
                ax.annotate('', xy=(mean_x[us_len], mean_y[us_len]), 
                    xytext=(mean_x[us_len-1], mean_y[us_len-1]),
                    arrowprops=dict(arrowstyle="->", color='black', lw=0.5), zorder=4) 
                
        x_limit = 1.5
        ax.set_xlim(-0.5, x_limit)
        ax.set_ylim(-0.05, 1.5) 
        ax.plot([-0.5, x_limit], [0, 0], color='black', linestyle='--', linewidth=0.75, alpha=0.5, zorder=1)
        
        ax.tick_params(axis='both', length=2, pad=1)
        ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(1))
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(0.5)
        ax.spines['bottom'].set_linewidth(0.5)
        
        if ax_idx == 0:
            ax.set_ylabel('Orthogonal stray dist.', labelpad=1)
        
        if ax_idx == 2:
            ax.legend(frameon=False, loc='upper right')
        
        if ax_idx == n_panels // 2:
            ax.set_xlabel('Position along association axis (CS=0, US=1)', labelpad=1)

    # 7. EXPORT
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)   
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    with open(f"{base_path}.json", "w") as f:
        json.dump(metrics_log, f, indent=4)

## 4.2 Quantification of the robust peak of returning pattern--CA1

In [ ]:
# 3D plotting with residuals for CA1
group_name = ['02.CA1-C', '03.CA1-I']
group_size = len(group_name) # 3
group_key = ['CA1-C', 'CA1-I']

base_du = 20 
post_du = 3+6
epochs = {
    'base':  (0,   int(base_du*fs)),
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du)*fs)) # Combining Post1 and Post2 for visual simplicity
}
epoch_durs = {'base':20, 'cs': 20, 'trace': 20, 'us': 3, 'pus': post_du}
epoch_colors = {'base': 'gray', 'cs': '#1f77b4', 'trace': '#2ca02c', 'us': '#d62728', 'pus': 'purple'}   

bins_plot_start = int((20-base_du)*fs) 
bins_plot_end = int(sum(epoch_durs.values())*fs) # 

# PLot with residuals
test_algori_data = 'post_02_2_resp_cal_new_z_residual'

data_3d_group = {}
for i in range(group_size):    
    dpath_test = os.path.join(dpath_cal_all, test_algori_data)
    print(group_name[i])  
    with open(os.path.join(dpath_test, f'{group_name[i]}_ds_cal_animal_wise.pkl'), 'rb') as f:
            ds_data = pickle.load(f)  
    
    # All eligible cells   
    Sig_bin_trial = []
    for animal_i in range(len(ds_data['whole'])):
        Sig_bin_trial.append(ds_data['whole'][animal_i][:, :, bins_plot_start: bins_plot_end])
    print(len(Sig_bin_trial))
    data_3d_group[group_key[i]] = Sig_bin_trial
        
width_mm = 40  # 
height_mm = 30
colors = ['#4091cf', '#e1703c', '#8cba54']
trials_to_plot = [1,2,6] 
plot_pca_phase_portrait_stat_jackknife(data_3d_group, trials_to_plot, epochs, group_key, fs, width_mm, height_mm, colors, dpath_plot, 
                     f'04_2_phase_portrait_stat_all trials_all_eligible_cells_residuals_jackknife')
                            
print('All finished************')         

In [ ]:
def add_significance_bar(ax, x1, x2, y_max, text, data_range):
    if text == 'ns' or not text: return y_max        
    
    line_gap = data_range * 0.05       
    text_offset = data_range * 0.02    
    star_height = data_range * 0.06

    y_line = y_max + line_gap
    ax.plot([x1, x2], [y_line, y_line], lw=0.75, c='k')   
    ax.text((x1+x2)*0.5, y_line + text_offset, text, ha='center', va='center', color='k', fontsize=7)
    
    return y_line + text_offset + star_height

def plot_pca_phase_portrait_stat_jackknife(data_3d_group, trials_to_plot, epochs, group_keys, fs, width_mm, height_mm, colors, output_path, title):
    """
    Quantifies the return amplitude of the pUS phase trajectory with strict QC.
    Uses Jackknife Resampling (Master PCA -> Leave-One-Animal-Out Projection).
    Plots ONLY the Robust Minimum (10th percentile) for the provided trials.
    """
    try: set_pub_style()
    except NameError: pass

    # 1. DEFINE EPOCHS
    cs_s, cs_e = epochs['cs']
    us_s, us_e = epochs['us']
    pus_e = epochs['pus'][1]
    
    us_len = us_e - us_s
    
    metrics_dict = {
        "Figure_Title": title,
        "Trials": trials_to_plot,
        "Artifact_Rejections": {grp: 0 for grp in group_keys},
        "Quantification": {grp: {t: {'robust_min': []} for t in trials_to_plot} for grp in group_keys}
    }
    
    # 2. EXTRACT & QUANTIFY DATA WITH JACKKNIFE PCA & QC
    for grp_idx, grp_name in enumerate(group_keys):
        if grp_name not in data_3d_group: continue
        
        # --- NEW: MASTER PCA FOR JACKKNIFE ---
        valid_animals = [a for a in data_3d_group[grp_name] if a is not None and a.size > 0 and a.shape[1] >= 3]
        if not valid_animals:
            continue
            
        animal_cell_counts = [a.shape[1] for a in valid_animals]
        n_trials, _, n_bins = valid_animals[0].shape
        
        # Pool all valid cells for the Master PCA
        master_data = np.concatenate(valid_animals, axis=1) 
        master_smoothed = gaussian_filter1d(master_data, sigma=2, axis=2)
        master_concat = np.transpose(master_smoothed, (1, 0, 2)).reshape(master_data.shape[1], -1)
        
        pca = PCA(n_components=3)
        pca.fit(master_concat.T)
        master_weights = pca.components_ 
        
        current_idx = 0
        for a_idx, n_c in enumerate(animal_cell_counts):
            # Jackknife: Identify indices excluding the current animal
            pseudo_idx = list(range(0, current_idx)) + list(range(current_idx + n_c, master_data.shape[1]))
            
            pseudo_data = master_smoothed[:, pseudo_idx, :]
            pseudo_weights = master_weights[:, pseudo_idx]
            
            pseudo_concat = np.transpose(pseudo_data, (0, 2, 1)).reshape(-1, len(pseudo_idx))
            correction_factor = master_data.shape[1] / len(pseudo_idx)
            
            # Project Pseudo-Population into Master PCA space
            pseudo_pcs = np.dot(pseudo_concat, (pseudo_weights * correction_factor).T)
            pca_trials = pseudo_pcs.reshape(n_trials, n_bins, 3)
            
            current_idx += n_c
            
            # --- DOWNSTREAM QUANTIFICATION & QC ---
            for t_num in trials_to_plot:
                t_idx = t_num - 1
                if t_idx >= n_trials: continue
                
                t_pcs = pca_trials[t_idx]
                
                cs_centroid = np.mean(t_pcs[cs_s:cs_e, :], axis=0)
                traj_eval_us = t_pcs[us_s:us_e, :]
                
                # Find US peak to define the vector
                dists_to_cs_us = np.linalg.norm(traj_eval_us - cs_centroid, axis=1)
                max_dist_idx = np.argmax(dists_to_cs_us)
                traj_1 = traj_eval_us[:max_dist_idx + 1, :]
                us_centroid = np.mean(traj_1, axis=0)
                
                vector_cs_us = us_centroid - cs_centroid
                vec_length = np.linalg.norm(vector_cs_us)
                
                # QC 1: Vector too small (Prevents division by near-zero)
                if vec_length < 1.0:
                    metrics_dict["Artifact_Rejections"][grp_name] += 1
                    continue
                
                # Project the pUS epoch onto the vector
                traj_pus = t_pcs[us_e:pus_e, :]
                V_pus = traj_pus - cs_centroid
                X_pus = np.dot(V_pus, vector_cs_us) / (vec_length ** 2)
                
                robust_min = np.percentile(X_pus, 10)
                
                # QC 2: Extreme Outlier Rejection
                if not np.isfinite(robust_min) or abs(robust_min) > 5.0:
                    metrics_dict["Artifact_Rejections"][grp_name] += 1
                    continue
                
                metrics_dict["Quantification"][grp_name][t_num]['robust_min'].append(float(robust_min))

    # 3. SET UP THE PLOT
    metric_to_plot = 'robust_min'
    y_axis_name = 'pUS peak returning dis.'
    
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    n_trials_plotted = len(trials_to_plot)
    x_positions = np.arange(1, n_trials_plotted + 1) * 1.5
    
    c_dark, c_light = colors[0], (lighten_color(colors[0], 0.4) if 'lighten_color' in globals() else colors[0])
    i_dark, i_light = colors[1], (lighten_color(colors[1], 0.4) if 'lighten_color' in globals() else colors[1])
    
    all_vals = []
    xtick_locs = []
    xtick_labels = []

    metrics_dict["Statistics_MWU"] = {}

    for t_idx, t_num in enumerate(trials_to_plot):
        x_center = x_positions[t_idx]
        x1, x2 = x_center - 0.2, x_center + 0.2
        
        # Only create one tick centered between the two groups
        xtick_locs.append(x_center)
        xtick_labels.append(f'T{t_num}')
        
        d1 = np.array(metrics_dict["Quantification"][group_keys[0]][t_num][metric_to_plot])
        d2 = np.array(metrics_dict["Quantification"][group_keys[1]][t_num][metric_to_plot])
        
        d1 = d1[~np.isnan(d1)]
        d2 = d2[~np.isnan(d2)]
        all_vals.extend(d1)
        all_vals.extend(d2)
        
        m1, s1 = np.mean(d1), np.std(d1) / np.sqrt(len(d1)) if len(d1) > 0 else (np.nan, np.nan)
        m2, s2 = np.mean(d2), np.std(d2) / np.sqrt(len(d2)) if len(d2) > 0 else (np.nan, np.nan)
        
        jitter = 0.1
        # Provide labels only on the first loop iteration so the legend doesn't duplicate
        l1 = group_keys[0] if t_idx == 0 else None
        l2 = group_keys[1] if t_idx == 0 else None
        
        ax.scatter(x1 + np.random.uniform(-jitter, jitter, size=len(d1)), d1, s=4.0, color=c_light, alpha=0.8, edgecolors='none', zorder=1, label=l1)
        ax.scatter(x2 + np.random.uniform(-jitter, jitter, size=len(d2)), d2, s=4.0, color=i_light, alpha=0.8, edgecolors='none', zorder=1, label=l2)
        
        if not np.isnan(m1):
            ax.errorbar(x1, m1, yerr=s1, fmt='o', color=c_dark, elinewidth=0.75, capsize=0, markersize=4, markeredgecolor='white', markeredgewidth=0.5, zorder=3)
        if not np.isnan(m2):
            ax.errorbar(x2, m2, yerr=s2, fmt='o', color=i_dark, elinewidth=0.75, capsize=0, markersize=4, markeredgecolor='white', markeredgewidth=0.5, zorder=3)

        # Statistics
        if len(d1) > 2 and len(d2) > 2:
            _, p_val = stats.mannwhitneyu(d1, d2, alternative='two-sided')
            metrics_dict["Statistics_MWU"][f"T{t_num}_{metric_to_plot}"] = float(p_val)
            
            star = get_asterisks(p_val)
            if star != 'ns':
                data_range = max(all_vals) - min(all_vals) if all_vals else 1.0
                local_max = max(np.max(d1), np.max(d2))
                add_significance_bar(ax, x1, x2, local_max, star, data_range)

    # 4. AXES FORMATTING WITH EMPTY DATA SAFETY
    ax.set_xticks(xtick_locs)
    ax.set_xticklabels(xtick_labels, fontsize=7)
    ax.set_ylabel(y_axis_name, labelpad=1)
    
    # Add a horizontal line at 0 (the CS baseline)
    ax.axhline(0, color='black', linestyle='--', linewidth=0.5, alpha=0.5, zorder=0)
    
    if all_vals:
        data_range = max(all_vals) - min(all_vals)
        if data_range == 0: data_range = 1.0
        # Give enough padding on x-axis so first/last points don't hit the spine
        ax.set_xlim(min(xtick_locs) - 0.7, max(xtick_locs) + 0.7)
        ax.set_ylim(min(all_vals) - data_range*0.1, max(all_vals) + data_range*0.3)
    else:
        # Fallback if 100% of animals were rejected for artifacts
        ax.set_xlim(0, 3)
        ax.set_ylim(-1, 2)
    
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    ax.tick_params(axis='both', length=2, pad=1)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_linewidth(0.5)
    
    # Add minimalist legend at top left
    #ax.legend(frameon=False, fontsize=5, loc='upper left', bbox_to_anchor=(0.0, 1.15), ncol=2)

    # 5. EXPORT
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, f"{title.replace(' ', '_')}_Stats")
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metrics_dict, output_path, f"{title}_Stats")

## 4.3 Group comparison of phase_portrait to the association axis---for EC3

In [ ]:
# 3D plotting with residuals for EC3
group_name = ['05.EC3-C', '06.EC3-I']
group_size = len(group_name) 
group_key = ['EC3-C', 'EC3-I']

base_du = 20 
post_du = 3+6 
epochs = {
    'base':  (0,   int(base_du*fs)),
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du)*fs)) 
}
epoch_durs = {'base':20, 'cs': 20, 'trace': 20, 'us': 3, 'pus': post_du}
epoch_colors = {'base': 'gray', 'cs': '#1f77b4', 'trace': '#2ca02c', 'us': '#d62728', 'pus': 'purple'}   

bins_plot_start = int((20-base_du)*fs) # pre-shock 3s
bins_plot_end = int(sum(epoch_durs.values())*fs) # 

# PLot with residuals
test_algori_data = 'post_02_2_resp_cal_new_z_residual'

data_3d_group = {}
for i in range(group_size):    
    dpath_test = os.path.join(dpath_cal_all, test_algori_data)
    print(group_name[i])  
    with open(os.path.join(dpath_test, f'{group_name[i]}_ds_cal_animal_wise.pkl'), 'rb') as f:
            ds_data = pickle.load(f)  
    
    # All eligible cells   
    Sig_bin_trial = []
    for animal_i in range(len(ds_data['whole'])):
        #if ds_data['whole'][animal_i].shape[1] > 300:
        Sig_bin_trial.append(ds_data['whole'][animal_i][:, :, bins_plot_start: bins_plot_end])
    print(len(Sig_bin_trial))
    data_3d_group[group_name[i]] = Sig_bin_trial
    
width_mm = 80  # 
height_mm = 30

colors = ['#4091cf', '#e1703c', '#8cba54']
trials_to_plot = [1,2,6] 

plot_pca_phase_portrait_groups_jackknife(data_3d_group, trials_to_plot, epochs, group_key, fs, width_mm, height_mm, colors, dpath_plot, 
                     f'04_3_phase_portrait_stat_all trials_all_eligible_cells_residuals-jackknife_EC3')                           
print('All finished************')         

In [ ]:

def plot_pca_phase_portrait_groups_jackknife(data_3d_group, trials_to_plot, epochs, group_key, fs, width_mm, height_mm, colors, output_path, title):
    """
    Jackknifed PCA Phase Portraits with Root-Cause Denoising.
    Method:
      1. Pools group cells for a Master PCA, uses Jackknife resampling for projection.
      2. Rejects trials where vec_length < 1.0 (insufficient state separation).
      3. Uses mean plotting
    """
    try: set_pub_style()
    except NameError: pass

    # 1. DEFINE EPOCH BOUNDARIES FOR EVALUATION
    cs_s, cs_e = epochs['cs']
    us_s = epochs['us'][0]
    us_e = epochs['us'][1]
    pus_e = epochs['pus'][1]
    
    us_len = us_e - us_s
    
    # 2. EXTRACT DATA WITH ROOT-CAUSE ARTIFACT REJECTION
    group_stats_raw = {}
    artifact_log = {grp: 0 for grp in data_3d_group.keys()}
    
    for grp_name, animals_ls in data_3d_group.items():
        group_stats_raw[grp_name] = {t_num: {"X": [], "Y": []} for t_num in trials_to_plot}
        
        # --- NEW: MASTER PCA FOR JACKKNIFE ---
        valid_animals = [a for a in animals_ls if a is not None and a.size > 0 and a.shape[1] >= 3]
        if not valid_animals:
            continue
            
        animal_cell_counts = [a.shape[1] for a in valid_animals]
        n_trials, _, n_bins = valid_animals[0].shape
        
        # Pool all valid cells for the Master PCA
        master_data = np.concatenate(valid_animals, axis=1) 
        master_smoothed = gaussian_filter1d(master_data, sigma=2, axis=2)
        master_concat = np.transpose(master_smoothed, (1, 0, 2)).reshape(master_data.shape[1], -1)
        
        pca = PCA(n_components=3)
        pca.fit(master_concat.T)
        master_weights = pca.components_ 
        
        current_idx = 0
        for a_idx, n_c in enumerate(animal_cell_counts):
            # Jackknife: Identify indices excluding the current animal
            pseudo_idx = list(range(0, current_idx)) + list(range(current_idx + n_c, master_data.shape[1]))
            
            pseudo_data = master_smoothed[:, pseudo_idx, :]
            pseudo_weights = master_weights[:, pseudo_idx]
            
            pseudo_concat = np.transpose(pseudo_data, (0, 2, 1)).reshape(-1, len(pseudo_idx))
            correction_factor = master_data.shape[1] / len(pseudo_idx)
            
            # Project Pseudo-Population into Master PCA space
            pseudo_pcs = np.dot(pseudo_concat, (pseudo_weights * correction_factor).T)
            pca_trials = pseudo_pcs.reshape(n_trials, n_bins, 3)
            
            current_idx += n_c
            
            # --- DOWNSTREAM PROCESSING (UNMODIFIED) ---
            for t_num in trials_to_plot:
                t_idx = t_num - 1
                if t_idx >= n_trials: continue
                
                t_pcs = pca_trials[t_idx]
                
                cs_centroid = np.mean(t_pcs[cs_s:cs_e, :], axis=0)
                traj_eval_us = t_pcs[us_s:us_e, :]
                dists_to_cs_us = np.linalg.norm(traj_eval_us - cs_centroid, axis=1)
                max_dist_idx = np.argmax(dists_to_cs_us)
                
                traj_1 = traj_eval_us[:max_dist_idx + 1, :]
                us_centroid = np.mean(traj_1, axis=0)
                
                vector_cs_us = us_centroid - cs_centroid
                vec_length = np.linalg.norm(vector_cs_us)
                
                # --- ROOT-CAUSE DENOISING TIER 1 ---
                if vec_length < 1.0:
                    artifact_log[grp_name] += 1
                    continue
                
                traj_eval_all = t_pcs[us_s:pus_e, :]
                V = traj_eval_all - cs_centroid
                X = np.dot(V, vector_cs_us) / (vec_length ** 2)
    
                V_parallel = np.outer(X, vector_cs_us)
                V_ortho = V - V_parallel
                Y = np.linalg.norm(V_ortho, axis=1) / vec_length # with normalization
                
                # Median filter to erase single-bin transient spikes
                X = median_filter(X, size=3)
                Y = median_filter(Y, size=3)
                
                group_stats_raw[grp_name][t_num]["X"].append(X)
                group_stats_raw[grp_name][t_num]["Y"].append(Y)

    # 3. METRICS EXTRACTION
    group_stats = group_stats_raw 
    metrics_log = {
        "Figure_Title": title,
        "Trials": trials_to_plot,
        "Quantification": {}
    }
    
    plot_limits = {t_num: {"x_min": 0, "x_max": 1, "y_max": 1} for t_num in trials_to_plot}
    
    for t_num in trials_to_plot:
        pooled_x, pooled_y = [], []
        for grp_name in group_stats.keys():
            if len(group_stats[grp_name][t_num]["X"]) > 0:
                pooled_x.extend(np.concatenate(group_stats[grp_name][t_num]["X"]))
                pooled_y.extend(np.concatenate(group_stats[grp_name][t_num]["Y"]))
                
        if len(pooled_x) > 0:
            plot_limits[t_num]["x_min"] = np.percentile(pooled_x, 10)
            plot_limits[t_num]["x_max"] = np.percentile(pooled_x, 90)
            plot_limits[t_num]["y_max"] = np.percentile(pooled_y, 90) 
            
            for grp_name in group_stats.keys():
                if grp_name not in metrics_log["Quantification"]:
                    metrics_log["Quantification"][grp_name] = {t: {"US_VarRatio": [], "pUS_VarRatio": [], "US_AUC": [], "pUS_AUC": []} for t in trials_to_plot}
                
                for i in range(len(group_stats[grp_name][t_num]["X"])):
                    clean_X = group_stats[grp_name][t_num]["X"][i]
                    clean_Y = group_stats[grp_name][t_num]["Y"][i]
                    
                    X_us, Y_us = clean_X[:us_len], clean_Y[:us_len]
                    X_pus, Y_pus = clean_X[us_len:], clean_Y[us_len:]
                    
                    vr_us = np.var(Y_us) / (np.var(X_us) + 1e-8)
                    vr_pus = np.var(Y_pus) / (np.var(X_pus) + 1e-8)
                    auc_us = np.sum(Y_us) / fs
                    auc_pus = np.sum(Y_pus) / fs
                    
                    metrics_log["Quantification"][grp_name][t_num]["US_VarRatio"].append(float(vr_us))
                    metrics_log["Quantification"][grp_name][t_num]["pUS_VarRatio"].append(float(vr_pus))
                    metrics_log["Quantification"][grp_name][t_num]["US_AUC"].append(float(auc_us))
                    metrics_log["Quantification"][grp_name][t_num]["pUS_AUC"].append(float(auc_pus))

    # 4. STATISTICAL TESTING (Mann-Whitney U)
    group_names = list(data_3d_group.keys())
    metrics_log["Statistics_MWU"] = {}
    
    if len(group_names) >= 2:
        g1, g2 = group_names[0], group_names[1]
        metrics_log["Statistics_MWU"][f"{g1}_vs_{g2}"] = {}
        
        for t_num in trials_to_plot:
            metrics_log["Statistics_MWU"][f"{g1}_vs_{g2}"][f"Trial_{t_num}"] = {}
            for metric in ["US_VarRatio", "pUS_VarRatio", "US_AUC", "pUS_AUC"]:
                if t_num in metrics_log["Quantification"][g1]:
                    data1 = metrics_log["Quantification"][g1][t_num][metric]
                    data2 = metrics_log["Quantification"][g2][t_num][metric]
                    
                    if len(data1) > 2 and len(data2) > 2:
                        _, p_val = stats.mannwhitneyu(data1, data2, alternative='two-sided')
                        metrics_log["Statistics_MWU"][f"{g1}_vs_{g2}"][f"Trial_{t_num}"][metric] = float(p_val)
                    else:
                        metrics_log["Statistics_MWU"][f"{g1}_vs_{g2}"][f"Trial_{t_num}"][metric] = "Not enough data"

    # 5. SET UP THE PLOT
    n_panels = len(trials_to_plot)
    fig, axes = plt.subplots(1, n_panels, 
                             figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    if n_panels == 1: axes = [axes]
    color_map = {grp: colors[i % len(colors)] for i, grp in enumerate(group_names)}

    # 6. PLOT PHASE PORTRAITS
    for ax_idx, t_num in enumerate(trials_to_plot):
        ax = axes[ax_idx]
        
        pad_x = (plot_limits[t_num]["x_max"] - plot_limits[t_num]["x_min"]) * 0.1
        ax_x_min = plot_limits[t_num]["x_min"] - pad_x
        ax_x_max = plot_limits[t_num]["x_max"] + pad_x
        
        ax.scatter([0, 1], [0, 0], color='black', s=10, zorder=2) 
        mean_x_max = 0
        for idx, grp_name in enumerate(group_names):
            if grp_name not in group_stats: continue
            list_x = group_stats[grp_name][t_num]["X"]
            list_y = group_stats[grp_name][t_num]["Y"]
            if len(list_x) == 0: continue
            
            for anim_x, anim_y in zip(list_x, list_y):
                smooth_x = gaussian_filter1d(anim_x, sigma=1.0)
                smooth_y = gaussian_filter1d(anim_y, sigma=1.0)
                ax.plot(smooth_x, smooth_y, color=color_map[grp_name], alpha=0.15, lw=0.5, zorder=3)
            
            mean_x = gaussian_filter1d(np.nanmean(np.array(list_x), axis=0), sigma=1.0)
            mean_y = gaussian_filter1d(np.nanmean(np.array(list_y), axis=0), sigma=1.0)
            ax.plot(mean_x, mean_y, color=color_map[grp_name], lw=0.75, label=group_key[idx], zorder=4)
            if np.max(mean_x) > mean_x_max:
                mean_x_max = np.max(mean_x)
            if us_len < len(mean_x):
                ax.annotate('', xy=(mean_x[us_len], mean_y[us_len]), 
                    xytext=(mean_x[us_len-1], mean_y[us_len-1]),
                    arrowprops=dict(arrowstyle="->", color='black', lw=0.5), zorder=4) 
                
        if mean_x_max < 2.0:
            x_limit = 2.0
        else:
            x_limit = 3.0
        ax.set_xlim(-0.5, x_limit)
        ax.set_ylim(-0.05, 3.0) #max(1.0, plot_limits[t_num]["y_max"] * 1.1)
        ax.plot([-0.5, x_limit], [0, 0], color='black', linestyle='--', linewidth=0.75, alpha=0.5, zorder=1)
        
        #x_limit = 1.5
        #ax.set_xlim(-0.5, x_limit)
        #ax.set_ylim(-0.05, 1.5) 
        #ax.plot([-0.5, x_limit], [0, 0], color='black', linestyle='--', linewidth=0.75, alpha=0.5, zorder=1)
        
        ax.tick_params(axis='both', length=2, pad=1)
        ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(1))
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(0.5)
        ax.spines['bottom'].set_linewidth(0.5)
        
        if ax_idx == 0:
            ax.set_ylabel('Orthogonal stray dist.', labelpad=1)
        
        if ax_idx == 2:
            ax.legend(frameon=False, loc='upper right')
        
        if ax_idx == n_panels // 2:
            ax.set_xlabel('Position along association axis (CS=0, US=1)', labelpad=1)

    # 7. EXPORT
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)   
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    with open(f"{base_path}.json", "w") as f:
        json.dump(metrics_log, f, indent=4)

## 4.4  Quantification of the peak of returning pattern--EC3

In [ ]:
# 3D plotting with residuals for EC3
group_name = ['05.EC3-C', '06.EC3-I']
group_size = len(group_name) 
group_key = ['EC3-C', 'EC3-I']

base_du = 20 
post_du = 3+6 
epochs = {
    'base':  (0,   int(base_du*fs)),
    'cs':  (int(base_du*fs),  int((base_du+20)*fs)),
    'trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'us': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pus':  (int((base_du+20+20+3)*fs), int((base_du+20+20+3+post_du)*fs))
}
epoch_durs = {'base':20, 'cs': 20, 'trace': 20, 'us': 3, 'pus': post_du}
epoch_colors = {'base': 'gray', 'cs': '#1f77b4', 'trace': '#2ca02c', 'us': '#d62728', 'pus': 'purple'}   

bins_plot_start = int((20-base_du)*fs) # pre-shock 3s
bins_plot_end = int(sum(epoch_durs.values())*fs) # 

# PLot with residuals
test_algori_data = 'post_02_2_resp_cal_new_z_residual'

data_3d_group = {}
for i in range(group_size):    
    dpath_test = os.path.join(dpath_cal_all, test_algori_data)
    print(group_name[i])  
    with open(os.path.join(dpath_test, f'{group_name[i]}_ds_cal_animal_wise.pkl'), 'rb') as f:
            ds_data = pickle.load(f)  
    
    # All eligible cells   
    Sig_bin_trial = []
    for animal_i in range(len(ds_data['whole'])):
        #if ds_data['whole'][animal_i].shape[1] > 300:
        Sig_bin_trial.append(ds_data['whole'][animal_i][:, :, bins_plot_start: bins_plot_end])
    print(len(Sig_bin_trial))
    data_3d_group[group_key[i]] = Sig_bin_trial
    
width_mm = 40  # 
height_mm = 30

colors = ['#4091cf', '#e1703c', '#8cba54']
trials_to_plot = [1,2,6] 

plot_pca_phase_portrait_stat_jackknife(data_3d_group, trials_to_plot, epochs, group_key, fs, width_mm, height_mm, colors, dpath_plot, 
                     f'04_4_phase_portrait_stat_all trials_all_eligible_cells_residuals_jackknife_EC3')                           
print('All finished************')         

In [ ]:
def add_significance_bar(ax, x1, x2, y_max, text, data_range):
    if text == 'ns' or not text: return y_max        
    
    line_gap = data_range * 0.05       
    text_offset = data_range * 0.02    
    star_height = data_range * 0.06

    y_line = y_max + line_gap
    ax.plot([x1, x2], [y_line, y_line], lw=0.75, c='k')   
    ax.text((x1+x2)*0.5, y_line + text_offset, text, ha='center', va='center', color='k', fontsize=7)
    
    return y_line + text_offset + star_height

def plot_pca_phase_portrait_stat_jackknife(data_3d_group, trials_to_plot, epochs, group_keys, fs, width_mm, height_mm, colors, output_path, title):
    """
    Quantifies the return amplitude of the pUS phase trajectory with strict QC.
    Uses Jackknife Resampling (Master PCA -> Leave-One-Animal-Out Projection).
    Plots ONLY the Robust Minimum (10th percentile) for the provided trials.
    """
    try: set_pub_style()
    except NameError: pass

    # 1. DEFINE EPOCHS
    cs_s, cs_e = epochs['cs']
    us_s, us_e = epochs['us']
    pus_e = epochs['pus'][1]
    
    us_len = us_e - us_s
    
    metrics_dict = {
        "Figure_Title": title,
        "Trials": trials_to_plot,
        "Artifact_Rejections": {grp: 0 for grp in group_keys},
        "Quantification": {grp: {t: {'robust_min': []} for t in trials_to_plot} for grp in group_keys}
    }
    
    # 2. EXTRACT & QUANTIFY DATA WITH JACKKNIFE PCA & QC
    for grp_idx, grp_name in enumerate(group_keys):
        if grp_name not in data_3d_group: continue
        
        # --- NEW: MASTER PCA FOR JACKKNIFE ---
        valid_animals = [a for a in data_3d_group[grp_name] if a is not None and a.size > 0 and a.shape[1] >= 3]
        if not valid_animals:
            continue
            
        animal_cell_counts = [a.shape[1] for a in valid_animals]
        n_trials, _, n_bins = valid_animals[0].shape
        
        # Pool all valid cells for the Master PCA
        master_data = np.concatenate(valid_animals, axis=1) 
        master_smoothed = gaussian_filter1d(master_data, sigma=2, axis=2)
        master_concat = np.transpose(master_smoothed, (1, 0, 2)).reshape(master_data.shape[1], -1)
        
        pca = PCA(n_components=3)
        pca.fit(master_concat.T)
        master_weights = pca.components_ 
        
        current_idx = 0
        for a_idx, n_c in enumerate(animal_cell_counts):
            # Jackknife: Identify indices excluding the current animal
            pseudo_idx = list(range(0, current_idx)) + list(range(current_idx + n_c, master_data.shape[1]))
            
            pseudo_data = master_smoothed[:, pseudo_idx, :]
            pseudo_weights = master_weights[:, pseudo_idx]
            
            pseudo_concat = np.transpose(pseudo_data, (0, 2, 1)).reshape(-1, len(pseudo_idx))
            correction_factor = master_data.shape[1] / len(pseudo_idx)
            
            # Project Pseudo-Population into Master PCA space
            pseudo_pcs = np.dot(pseudo_concat, (pseudo_weights * correction_factor).T)
            pca_trials = pseudo_pcs.reshape(n_trials, n_bins, 3)
            
            current_idx += n_c
            
            # --- DOWNSTREAM QUANTIFICATION & QC ---
            for t_num in trials_to_plot:
                t_idx = t_num - 1
                if t_idx >= n_trials: continue
                
                t_pcs = pca_trials[t_idx]
                
                cs_centroid = np.mean(t_pcs[cs_s:cs_e, :], axis=0)
                traj_eval_us = t_pcs[us_s:us_e, :]
                
                # Find US peak to define the vector
                dists_to_cs_us = np.linalg.norm(traj_eval_us - cs_centroid, axis=1)
                max_dist_idx = np.argmax(dists_to_cs_us)
                traj_1 = traj_eval_us[:max_dist_idx + 1, :]
                us_centroid = np.mean(traj_1, axis=0)
                
                vector_cs_us = us_centroid - cs_centroid
                vec_length = np.linalg.norm(vector_cs_us)
                
                # QC 1: Vector too small (Prevents division by near-zero)
                if vec_length < 1.0:
                    metrics_dict["Artifact_Rejections"][grp_name] += 1
                    continue
                
                # Project the pUS epoch onto the vector
                traj_pus = t_pcs[us_e:pus_e, :]
                V_pus = traj_pus - cs_centroid
                X_pus = np.dot(V_pus, vector_cs_us) / (vec_length ** 2)
                
                robust_min = np.percentile(X_pus, 10)
                
                # QC 2: Extreme Outlier Rejection
                if not np.isfinite(robust_min) or abs(robust_min) > 5.0:
                    metrics_dict["Artifact_Rejections"][grp_name] += 1
                    continue
                
                metrics_dict["Quantification"][grp_name][t_num]['robust_min'].append(float(robust_min))

    # 3. SET UP THE PLOT
    metric_to_plot = 'robust_min'
    y_axis_name = 'pUS peak returning dis.'
    
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    n_trials_plotted = len(trials_to_plot)
    x_positions = np.arange(1, n_trials_plotted + 1) * 1.5
    
    c_dark, c_light = colors[0], (lighten_color(colors[0], 0.4) if 'lighten_color' in globals() else colors[0])
    i_dark, i_light = colors[1], (lighten_color(colors[1], 0.4) if 'lighten_color' in globals() else colors[1])
    
    all_vals = []
    xtick_locs = []
    xtick_labels = []

    metrics_dict["Statistics_MWU"] = {}

    for t_idx, t_num in enumerate(trials_to_plot):
        x_center = x_positions[t_idx]
        x1, x2 = x_center - 0.2, x_center + 0.2
        
        # Only create one tick centered between the two groups
        xtick_locs.append(x_center)
        xtick_labels.append(f'T{t_num}')
        
        d1 = np.array(metrics_dict["Quantification"][group_keys[0]][t_num][metric_to_plot])
        d2 = np.array(metrics_dict["Quantification"][group_keys[1]][t_num][metric_to_plot])
        
        d1 = d1[~np.isnan(d1)]
        d2 = d2[~np.isnan(d2)]
        all_vals.extend(d1)
        all_vals.extend(d2)
        
        m1, s1 = np.mean(d1), np.std(d1) / np.sqrt(len(d1)) if len(d1) > 0 else (np.nan, np.nan)
        m2, s2 = np.mean(d2), np.std(d2) / np.sqrt(len(d2)) if len(d2) > 0 else (np.nan, np.nan)
        
        jitter = 0.15
        # Provide labels only on the first loop iteration so the legend doesn't duplicate
        l1 = group_keys[0] if t_idx == 0 else None
        l2 = group_keys[1] if t_idx == 0 else None
        
        ax.scatter(x1 + np.random.uniform(-jitter, jitter, size=len(d1)), d1, s=4.0, color=c_light, alpha=0.8, edgecolors='none', zorder=1, label=l1)
        ax.scatter(x2 + np.random.uniform(-jitter, jitter, size=len(d2)), d2, s=4.0, color=i_light, alpha=0.8, edgecolors='none', zorder=1, label=l2)
        
        if not np.isnan(m1):
            ax.errorbar(x1, m1, yerr=s1, fmt='o', color=c_dark, elinewidth=0.75, capsize=0, markersize=4, markeredgecolor='white', markeredgewidth=0.5, zorder=3)
        if not np.isnan(m2):
            ax.errorbar(x2, m2, yerr=s2, fmt='o', color=i_dark, elinewidth=0.75, capsize=0, markersize=4, markeredgecolor='white', markeredgewidth=0.5, zorder=3)

        # Statistics
        if len(d1) > 2 and len(d2) > 2:
            _, p_val = stats.mannwhitneyu(d1, d2, alternative='two-sided')
            metrics_dict["Statistics_MWU"][f"T{t_num}_{metric_to_plot}"] = float(p_val)
            
            star = get_asterisks(p_val)
            if star != 'ns':
                data_range = max(all_vals) - min(all_vals) if all_vals else 1.0
                local_max = max(np.max(d1), np.max(d2))
                add_significance_bar(ax, x1, x2, local_max, star, data_range)

    # 4. AXES FORMATTING WITH EMPTY DATA SAFETY
    ax.set_xticks(xtick_locs)
    ax.set_xticklabels(xtick_labels, fontsize=7)
    ax.set_ylabel(y_axis_name, labelpad=1)
    
    # Add a horizontal line at 0 (the CS baseline)
    ax.axhline(0, color='black', linestyle='--', linewidth=0.5, alpha=0.5, zorder=0)
    
    if all_vals:
        data_range = max(all_vals) - min(all_vals)
        if data_range == 0: data_range = 1.0
        # Give enough padding on x-axis so first/last points don't hit the spine
        ax.set_xlim(min(xtick_locs) - 0.7, max(xtick_locs) + 0.7)
        ax.set_ylim(min(all_vals) - data_range*0.1, max(all_vals) + data_range*0.3)
    else:
        # Fallback if 100% of animals were rejected for artifacts
        ax.set_xlim(0, 3)
        ax.set_ylim(-1, 2)
    
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    ax.tick_params(axis='both', length=2, pad=1)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_linewidth(0.5)
    
    # Add minimalist legend at top left
    #ax.legend(frameon=False, fontsize=5, loc='upper left', bbox_to_anchor=(0.0, 1.15), ncol=2)

    # 5. EXPORT
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, f"{title.replace(' ', '_')}_Stats")
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metrics_dict, output_path, f"{title}_Stats")